# Historical experiment: ResNet18 ordinal, hybrid, CORAL/CORN, and interpretability experiments

**Status:** exploratory archive, not the canonical pipeline used for the current manuscript results.

This notebook is preserved to document the development history. Its code and scientific logic are retained, but stored cell outputs and execution counts were removed to keep the repository lightweight and avoid presenting stale outputs as current results. Some cells may depend on Google Colab paths, checkpoints, variables created by earlier cells, or experimental configurations that are not included in this repository. It has not been rerun end-to-end as part of this cleanup.

For the current pipeline and manuscript-aligned evaluation, use [`../KOA_OrdinalConvNeXt_PCA_LinearSVC_Final.ipynb`](../KOA_OrdinalConvNeXt_PCA_LinearSVC_Final.ipynb).

In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT
# ============================================================

import os
import random
import numpy as np
import torch

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

print("Seed:", SEED)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 2 — DATASET
# ============================================================

import kagglehub

DATASET_PATH = kagglehub.dataset_download(
    "shashwatwork/knee-osteoarthritis-dataset-with-severity"
)

TRAIN_DIR = os.path.join(
    DATASET_PATH,
    "train"
)

VAL_DIR = os.path.join(
    DATASET_PATH,
    "val"
)

TEST_DIR = os.path.join(
    DATASET_PATH,
    "test"
)

print("=" * 70)
print("DATASET")
print("=" * 70)

print("Dataset:", DATASET_PATH)
print("Train:", os.path.exists(TRAIN_DIR))
print("Val:", os.path.exists(VAL_DIR))
print("Test:", os.path.exists(TEST_DIR))

print("=" * 70)

In [ ]:
# ============================================================
# CELL 3 — PREPROCESSING / TRANSFORMS
# ============================================================

import cv2
from PIL import Image
from torchvision import transforms


# ------------------------------------------------------------
# CLAHE transform
# ------------------------------------------------------------

class CLAHETransform:

    def __init__(
        self,
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):

        img = np.array(img)

        # Convert to grayscale if needed
        if img.ndim == 3:
            img = cv2.cvtColor(
                img,
                cv2.COLOR_RGB2GRAY
            )

        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )

        img = clahe.apply(img)

        return Image.fromarray(img)


# ------------------------------------------------------------
# Training transform
# ------------------------------------------------------------

train_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    # Conservative augmentation
    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=5
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Validation / Test transform
# ------------------------------------------------------------

eval_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

test_image = Image.new(
    "L",
    (512, 512),
    128
)

test_output = train_transform(
    test_image
)

print("=" * 70)
print("PREPROCESSING READY")
print("=" * 70)

print(
    "Train transform output:",
    test_output.shape
)

print(
    "Expected shape:",
    "torch.Size([3, 224, 224])"
)

print(
    "CLAHE clip limit:",
    2.0
)

print(
    "CLAHE grid:",
    (8, 8)
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 4 — DATASETS + REPRODUCIBLE WEIGHTED SAMPLER
# ============================================================

from torchvision import datasets
from torch.utils.data import (
    DataLoader,
    WeightedRandomSampler
)


# ------------------------------------------------------------
# 1. Reproducibility helpers
# ------------------------------------------------------------

def seed_worker(worker_id):

    worker_seed = (
        torch.initial_seed()
        % 2**32
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


data_generator = torch.Generator()
data_generator.manual_seed(SEED)


# ------------------------------------------------------------
# 2. Create datasets
# ------------------------------------------------------------

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform
)


# ------------------------------------------------------------
# 3. Class names
# ------------------------------------------------------------

class_names = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]


# ------------------------------------------------------------
# 4. Class counts
# ------------------------------------------------------------

train_targets = np.array(
    train_dataset.targets
)

class_counts = np.bincount(
    train_targets,
    minlength=5
)


# ------------------------------------------------------------
# 5. WeightedRandomSampler
# ------------------------------------------------------------

class_sampling_weights = (
    1.0 / class_counts
)

sample_weights = (
    class_sampling_weights[
        train_targets
    ]
)

sample_weights = torch.as_tensor(
    sample_weights,
    dtype=torch.double
)

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)


# ------------------------------------------------------------
# 6. DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=weighted_sampler,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=data_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# 7. Sanity check
# ------------------------------------------------------------

images, labels = next(
    iter(train_loader)
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("=" * 70)
print("DATASETS + SAMPLER READY")
print("=" * 70)

print("\nClass mapping:")
print(train_dataset.class_to_idx)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nTraining class counts:")

for i, count in enumerate(class_counts):

    print(
        f"KL{i} ({class_names[i]}): {count}"
    )

print("\nSampler weights:")

for i, weight in enumerate(
    class_sampling_weights
):

    print(
        f"KL{i}: {weight:.8f}"
    )

print("\nBatch shape:")
print(images.shape)

print("\nNumber of batches:")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

print("\nSeed:", SEED)
print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("=" * 70)

In [ ]:
# ============================================================
# CELL 5 — RESNET18 MODEL
# ============================================================

import torch.nn as nn
from torchvision import models


# ------------------------------------------------------------
# Build pretrained ResNet18
# ------------------------------------------------------------

model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Replace ImageNet classifier (1000 classes)
# with our 5 KL grades
model.fc = nn.Linear(
    model.fc.in_features,
    5
)

model = model.to(device)


# ------------------------------------------------------------
# Full fine-tuning
# ------------------------------------------------------------

for param in model.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# Count parameters
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 70)
print("RESNET18 MODEL READY")
print("=" * 70)

print(
    f"Total parameters     : {total_params:,}"
)

print(
    f"Trainable parameters : {trainable_params:,}"
)

print(
    "Architecture          : ResNet18"
)

print(
    "Pretrained            : ImageNet"
)

print(
    "Output classes        : 5"
)

print(
    "Input resolution      : 224x224"
)

print(
    "Fine-tuning           : Full model"
)

print(
    "Device                :",
    device
)

if torch.cuda.is_available():

    print(
        "GPU                   :",
        torch.cuda.get_device_name(0)
    )

print("=" * 70)

In [ ]:
# ============================================================
# CELL 6 — LOAD BEST CNN CHECKPOINT
# Prepare ResNet18 Backbone for Hybrid Model
# ============================================================

import os
import torch
import torch.nn as nn
from torchvision import models


# ------------------------------------------------------------
# 1. Checkpoint path
# ------------------------------------------------------------

CHECKPOINT_PATH = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT_PATH}\n\n"
        "Please upload best_resnet18_sampler_seed2026.pth "
        "to the Colab session."
    )


# ------------------------------------------------------------
# 2. Load checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)


# ------------------------------------------------------------
# 3. Build ResNet18
# ------------------------------------------------------------

resnet_backbone = models.resnet18(
    weights=None
)

resnet_backbone.fc = nn.Linear(
    resnet_backbone.fc.in_features,
    5
)


# ------------------------------------------------------------
# 4. Load trained weights
# ------------------------------------------------------------

resnet_backbone.load_state_dict(
    checkpoint["model_state_dict"]
)

resnet_backbone = resnet_backbone.to(
    device
)


# ------------------------------------------------------------
# 5. Remove final classification layer
#    Output = 512-dimensional deep feature
# ------------------------------------------------------------

deep_feature_extractor = nn.Sequential(
    *list(resnet_backbone.children())[:-1]
)


# ------------------------------------------------------------
# 6. Freeze CNN during initial Hybrid experiment
# ------------------------------------------------------------

for param in deep_feature_extractor.parameters():
    param.requires_grad = False

deep_feature_extractor.eval()


# ------------------------------------------------------------
# 7. Check
# ------------------------------------------------------------

print("=" * 70)
print("HYBRID BACKBONE READY")
print("=" * 70)

print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

print(
    "Starting Macro F1:",
    checkpoint["macro_f1"]
)

print(
    "Starting Val Accuracy:",
    checkpoint["val_acc"]
)

print(
    "Deep feature dimension: 512"
)

print(
    "CNN backbone frozen: True"
)

print(
    "Device:",
    device
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("=" * 70)

In [ ]:
# ============================================================
# CHECKPOINT INTEGRITY CHECK
# ============================================================

import os

CHECKPOINT_PATH = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

print("=" * 70)
print("CHECKPOINT INTEGRITY CHECK")
print("=" * 70)

if not os.path.exists(CHECKPOINT_PATH):

    print("❌ File not found")

else:

    size_bytes = os.path.getsize(
        CHECKPOINT_PATH
    )

    size_mb = (
        size_bytes /
        (1024 ** 2)
    )

    print("File found: ✅")
    print(
        f"File size: {size_mb:.2f} MB"
    )

    print(
        "Path:",
        CHECKPOINT_PATH
    )

print("=" * 70)

In [ ]:
# ============================================================
# CELL 6 — CHECKPOINT DEEP INTEGRITY TEST
# ============================================================

import os
import zipfile
import hashlib

CHECKPOINT_PATH = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

print("=" * 70)
print("CHECKPOINT DEEP INTEGRITY TEST")
print("=" * 70)


# ------------------------------------------------------------
# 1. Basic file information
# ------------------------------------------------------------

size_bytes = os.path.getsize(
    CHECKPOINT_PATH
)

print(
    f"File size: {size_bytes / (1024**2):.2f} MB"
)


# ------------------------------------------------------------
# 2. Check first bytes
# ------------------------------------------------------------

with open(
    CHECKPOINT_PATH,
    "rb"
) as f:

    header = f.read(8)

print(
    "First 8 bytes:",
    header
)


# ------------------------------------------------------------
# 3. SHA256
# ------------------------------------------------------------

sha256 = hashlib.sha256()

with open(
    CHECKPOINT_PATH,
    "rb"
) as f:

    for chunk in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):

        sha256.update(chunk)

print(
    "SHA256:",
    sha256.hexdigest()
)


# ------------------------------------------------------------
# 4. ZIP archive test
# PyTorch .pth files normally use a zip-based format
# ------------------------------------------------------------

print("\nZIP integrity:")

try:

    with zipfile.ZipFile(
        CHECKPOINT_PATH,
        "r"
    ) as zf:

        bad_file = zf.testzip()

        if bad_file is None:

            print(
                "✅ ZIP archive is readable"
            )

            print(
                "Number of archive files:",
                len(zf.namelist())
            )

        else:

            print(
                "❌ Corrupted archive entry:",
                bad_file
            )

except zipfile.BadZipFile as e:

    print(
        "❌ File is NOT a valid ZIP/PyTorch archive"
    )

    print(
        "Error:",
        e
    )


print("=" * 70)

In [ ]:
# ============================================================
# CELL 7 — Inspect PyTorch Checkpoint Archive
# ============================================================

import zipfile
import os

CHECKPOINT_PATH = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

print("=" * 70)
print("PYTORCH CHECKPOINT ARCHIVE INSPECTION")
print("=" * 70)

with zipfile.ZipFile(
    CHECKPOINT_PATH,
    "r"
) as zf:

    names = zf.namelist()

    print(
        f"Archive entries: {len(names)}"
    )

    print("\nFirst 30 entries:")

    for name in names[:30]:
        print(" ", name)

    print("\nImportant files:")

    important_keywords = [
        "data.pkl",
        "byteorder",
        "version",
        "serialization_id"
    ]

    for keyword in important_keywords:

        matches = [
            name
            for name in names
            if keyword in name
        ]

        print(
            f"{keyword}: {matches}"
        )

    print("\nCompression types:")

    compression_types = {}

    for info in zf.infolist():

        c = info.compress_type

        compression_types[c] = (
            compression_types.get(c, 0) + 1
        )

    print(
        compression_types
    )

print("=" * 70)

In [ ]:
# ============================================================
# CELL 8 — REPACK CHECKPOINT
# ============================================================

import os
import zipfile
import shutil
import torch

SOURCE = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

REPACKED = (
    "/content/best_resnet18_sampler_seed2026_repacked.pth"
)

print("=" * 70)
print("REPACKING CHECKPOINT")
print("=" * 70)


# ------------------------------------------------------------
# Rebuild a clean ZIP archive
# ------------------------------------------------------------

with zipfile.ZipFile(
    SOURCE,
    "r"
) as zin:

    with zipfile.ZipFile(
        REPACKED,
        "w",
        compression=zipfile.ZIP_STORED
    ) as zout:

        for item in zin.infolist():

            data = zin.read(
                item.filename
            )

            zout.writestr(
                item.filename,
                data
            )


# ------------------------------------------------------------
# Check new file
# ------------------------------------------------------------

source_size = os.path.getsize(
    SOURCE
) / (1024 ** 2)

repacked_size = os.path.getsize(
    REPACKED
) / (1024 ** 2)

print(
    f"Original size : {source_size:.2f} MB"
)

print(
    f"Repacked size : {repacked_size:.2f} MB"
)


# ------------------------------------------------------------
# Try loading repacked checkpoint
# ------------------------------------------------------------

print("\nTesting torch.load...")

try:

    checkpoint_test = torch.load(
        REPACKED,
        map_location="cpu",
        weights_only=False
    )

    print("✅ SUCCESS — checkpoint loaded!")

    print(
        "\nCheckpoint keys:"
    )

    print(
        list(checkpoint_test.keys())
    )

    if "model_state_dict" in checkpoint_test:

        print(
            "\nModel tensors:",
            len(
                checkpoint_test[
                    "model_state_dict"
                ]
            )
        )

    if "macro_f1" in checkpoint_test:

        print(
            "Macro F1:",
            checkpoint_test["macro_f1"]
        )

except Exception as e:

    print("❌ torch.load still failed")

    print(
        type(e).__name__,
        ":",
        str(e)
    )

print("=" * 70)

In [ ]:
# ============================================================
# CELL 9 — HYBRID DATASET
# X-ray + 25 Handcrafted ROI Features
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from PIL import Image
from torch.utils.data import Dataset, DataLoader


# ------------------------------------------------------------
# 1. Load saved ROI handcrafted features
# ------------------------------------------------------------

TRAIN_ROI_CSV = (
    "/content/train_roi_classical_features.csv"
)

VAL_ROI_CSV = (
    "/content/val_roi_classical_features.csv"
)

train_roi_df = pd.read_csv(
    TRAIN_ROI_CSV
)

val_roi_df = pd.read_csv(
    VAL_ROI_CSV
)


# ------------------------------------------------------------
# 2. Select the 25 handcrafted features
# ------------------------------------------------------------

roi_feature_columns = [
    col
    for col in train_roi_df.columns
    if col not in [
        "label",
        "image_path"
    ]
]

assert len(
    roi_feature_columns
) == 25


# ------------------------------------------------------------
# 3. Hybrid Dataset
# ------------------------------------------------------------

class HybridDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        # -----------------------------
        # Image
        # -----------------------------

        image_path = self.df.loc[
            idx,
            "image_path"
        ]

        image = Image.open(
            image_path
        ).convert("L")

        image = self.transform(
            image
        )

        # -----------------------------
        # Handcrafted features
        # -----------------------------

        handcrafted = self.df.loc[
            idx,
            roi_feature_columns
        ].values.astype(
            np.float32
        )

        handcrafted = torch.tensor(
            handcrafted,
            dtype=torch.float32
        )

        # -----------------------------
        # Label
        # -----------------------------

        label = torch.tensor(
            int(
                self.df.loc[
                    idx,
                    "label"
                ]
            ),
            dtype=torch.long
        )

        return (
            image,
            handcrafted,
            label
        )


# ------------------------------------------------------------
# 4. Build datasets
# ------------------------------------------------------------

train_hybrid_dataset = HybridDataset(
    train_roi_df,
    eval_transform
)

val_hybrid_dataset = HybridDataset(
    val_roi_df,
    eval_transform
)


# ------------------------------------------------------------
# 5. DataLoaders
# ------------------------------------------------------------

train_hybrid_loader = DataLoader(
    train_hybrid_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_hybrid_loader = DataLoader(
    val_hybrid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# 6. Sanity check
# ------------------------------------------------------------

images, handcrafted, labels = next(
    iter(train_hybrid_loader)
)


# ------------------------------------------------------------
# 7. Verify alignment
# ------------------------------------------------------------

assert images.shape == (
    32,
    3,
    224,
    224
)

assert handcrafted.shape == (
    32,
    25
)

assert labels.shape == (
    32,
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("=" * 70)
print("HYBRID DATASET READY")
print("=" * 70)

print(
    "Train samples:",
    len(train_hybrid_dataset)
)

print(
    "Validation samples:",
    len(val_hybrid_dataset)
)

print(
    "Image batch:",
    images.shape
)

print(
    "Handcrafted batch:",
    handcrafted.shape
)

print(
    "Label batch:",
    labels.shape
)

print(
    "Handcrafted features:",
    len(roi_feature_columns)
)

print(
    "\nFirst 5 handcrafted features:"
)

for name in roi_feature_columns[:5]:
    print(" ", name)

print("=" * 70)

In [ ]:
# ============================================================
# HYBRID SETUP — SELF-CONTAINED
# No CSV dependency
# Rebuild 25 ROI handcrafted features + verify CNN checkpoint
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from torchvision import datasets


# ============================================================
# 1. CHECK REQUIRED PATHS
# ============================================================

required_paths = {
    "TRAIN_DIR": TRAIN_DIR,
    "VAL_DIR": VAL_DIR,
}

for name, path in required_paths.items():

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )


# ============================================================
# 2. CHECK CNN CHECKPOINT
# ============================================================

CHECKPOINT_PATH = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        "\nCNN checkpoint is missing:\n"
        f"{CHECKPOINT_PATH}\n\n"
        "Upload best_resnet18_sampler_seed2026.pth "
        "to this Colab runtime first."
    )


# ============================================================
# 3. LOAD DATASET METADATA
# ============================================================

train_imagefolder = datasets.ImageFolder(
    TRAIN_DIR
)

val_imagefolder = datasets.ImageFolder(
    VAL_DIR
)

print("=" * 70)
print("HYBRID SETUP")
print("=" * 70)

print(
    "Train samples:",
    len(train_imagefolder)
)

print(
    "Validation samples:",
    len(val_imagefolder)
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH
)


# ============================================================
# 4. ROI FEATURE EXTRACTION
# ============================================================

def extract_roi_features(
    image_path,
    crop_ratio=0.80
):
    """
    Extract 25 handcrafted features from a center ROI.

    10 intensity/statistical
    5 GLCM
    10 LBP
    """

    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(
            f"Cannot read image:\n{image_path}"
        )

    # Fixed image size
    image = cv2.resize(
        image,
        (224, 224),
        interpolation=cv2.INTER_AREA
    )

    # --------------------------------------------------------
    # Center ROI
    # --------------------------------------------------------

    crop_size = int(
        224 * crop_ratio
    )

    start = (
        224 - crop_size
    ) // 2

    end = start + crop_size

    roi = image[
        start:end,
        start:end
    ]

    # --------------------------------------------------------
    # Intensity features — 10
    # --------------------------------------------------------

    features = [

        np.mean(roi),
        np.std(roi),
        np.min(roi),
        np.max(roi),

        np.percentile(roi, 10),
        np.percentile(roi, 25),
        np.percentile(roi, 50),
        np.percentile(roi, 75),
        np.percentile(roi, 90),
    ]

    # Entropy
    histogram = cv2.calcHist(
        [roi],
        [0],
        None,
        [256],
        [0, 256]
    ).flatten()

    histogram = histogram / (
        histogram.sum() + 1e-8
    )

    entropy = -np.sum(
        histogram *
        np.log2(histogram + 1e-10)
    )

    features.append(entropy)

    # --------------------------------------------------------
    # GLCM — 5
    # --------------------------------------------------------

    from skimage.feature import (
        graycomatrix,
        graycoprops
    )

    roi_32 = (
        roi / 8
    ).astype(np.uint8)

    glcm = graycomatrix(
        roi_32,
        distances=[1, 2],
        angles=[
            0,
            np.pi / 4,
            np.pi / 2,
            3 * np.pi / 4
        ],
        levels=32,
        symmetric=True,
        normed=True
    )

    for prop in [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation"
    ]:

        value = graycoprops(
            glcm,
            prop
        ).mean()

        features.append(value)

    # --------------------------------------------------------
    # LBP — 10
    # --------------------------------------------------------

    from skimage.feature import (
        local_binary_pattern
    )

    lbp = local_binary_pattern(
        roi,
        P=8,
        R=1,
        method="uniform"
    )

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, 11),
        range=(0, 10)
    )

    lbp_hist = lbp_hist.astype(
        np.float32
    )

    lbp_hist /= (
        lbp_hist.sum() + 1e-8
    )

    features.extend(
        lbp_hist.tolist()
    )

    features = np.asarray(
        features,
        dtype=np.float32
    )

    if len(features) != 25:
        raise RuntimeError(
            f"Expected 25 features, "
            f"got {len(features)}"
        )

    return features


# ============================================================
# 5. BUILD FEATURE ARRAYS
# ============================================================

def build_roi_feature_array(
    dataset,
    split_name
):

    all_features = []
    all_labels = []
    all_paths = []

    for image_path, label in tqdm(
        dataset.samples,
        desc=f"ROI features — {split_name}"
    ):

        features = extract_roi_features(
            image_path
        )

        all_features.append(
            features
        )

        all_labels.append(
            label
        )

        all_paths.append(
            image_path
        )

    return (
        np.stack(all_features),
        np.asarray(
            all_labels,
            dtype=np.int64
        ),
        all_paths
    )


# ============================================================
# 6. EXTRACT TRAIN + VALIDATION
# ============================================================

train_roi_features, train_roi_labels, train_roi_paths = (
    build_roi_feature_array(
        train_imagefolder,
        "TRAIN"
    )
)

val_roi_features, val_roi_labels, val_roi_paths = (
    build_roi_feature_array(
        val_imagefolder,
        "VALIDATION"
    )
)


# ============================================================
# 7. VERIFY
# ============================================================

assert train_roi_features.shape == (
    len(train_imagefolder),
    25
)

assert val_roi_features.shape == (
    len(val_imagefolder),
    25
)

assert np.array_equal(
    train_roi_labels,
    np.asarray(
        train_imagefolder.targets
    )
)

assert np.array_equal(
    val_roi_labels,
    np.asarray(
        val_imagefolder.targets
    )
)


# ============================================================
# 8. SAVE TO NPZ
# ============================================================

TRAIN_ROI_NPZ = (
    "/content/train_roi_features.npz"
)

VAL_ROI_NPZ = (
    "/content/val_roi_features.npz"
)

np.savez_compressed(
    TRAIN_ROI_NPZ,
    features=train_roi_features,
    labels=train_roi_labels
)

np.savez_compressed(
    VAL_ROI_NPZ,
    features=val_roi_features,
    labels=val_roi_labels
)


# ============================================================
# 9. CHECKPOINT TEST
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)


# ============================================================
# 10. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("HYBRID INPUTS READY")
print("=" * 70)

print(
    "CNN checkpoint Macro F1:",
    checkpoint["macro_f1"]
)

print(
    "Train ROI shape:",
    train_roi_features.shape
)

print(
    "Validation ROI shape:",
    val_roi_features.shape
)

print(
    "Train labels:",
    train_roi_labels.shape
)

print(
    "Validation labels:",
    val_roi_labels.shape
)

print(
    "\nSaved:"
)

print(TRAIN_ROI_NPZ)
print(VAL_ROI_NPZ)

print(
    "\n25 handcrafted features are now independent "
    "of CSV files."
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 10 — HYBRID NEURAL NETWORK SETUP
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from sklearn.preprocessing import StandardScaler
from torchvision import models


# ============================================================
# 1. Load ROI handcrafted features
# ============================================================

train_roi_data = np.load(
    "/content/train_roi_features.npz"
)

val_roi_data = np.load(
    "/content/val_roi_features.npz"
)

X_train_roi = train_roi_data["features"].astype(
    np.float32
)

y_train_roi = train_roi_data["labels"].astype(
    np.int64
)

X_val_roi = val_roi_data["features"].astype(
    np.float32
)

y_val_roi = val_roi_data["labels"].astype(
    np.int64
)


# ============================================================
# 2. Standardize handcrafted features
#    IMPORTANT: fit ONLY on training data
# ============================================================

roi_scaler = StandardScaler()

X_train_roi_scaled = roi_scaler.fit_transform(
    X_train_roi
).astype(np.float32)

X_val_roi_scaled = roi_scaler.transform(
    X_val_roi
).astype(np.float32)


# ============================================================
# 3. Verify label alignment
# ============================================================

assert np.array_equal(
    y_train_roi,
    np.asarray(train_dataset.targets)
)

assert np.array_equal(
    y_val_roi,
    np.asarray(val_dataset.targets)
)


# ============================================================
# 4. Hybrid Dataset
# ============================================================

class HybridDataset(Dataset):

    def __init__(
        self,
        image_dataset,
        handcrafted_features,
        labels
    ):

        self.image_dataset = image_dataset

        self.handcrafted_features = torch.tensor(
            handcrafted_features,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        image, _ = self.image_dataset[idx]

        handcrafted = (
            self.handcrafted_features[idx]
        )

        label = self.labels[idx]

        return (
            image,
            handcrafted,
            label
        )


# ============================================================
# 5. Create hybrid datasets
# ============================================================

train_hybrid_dataset = HybridDataset(
    train_dataset,
    X_train_roi_scaled,
    y_train_roi
)

val_hybrid_dataset = HybridDataset(
    val_dataset,
    X_val_roi_scaled,
    y_val_roi
)


# ============================================================
# 6. WeightedRandomSampler
#    Same sampling strategy as our best CNN
# ============================================================

train_targets_hybrid = np.asarray(
    train_hybrid_dataset.labels
)

hybrid_class_counts = np.bincount(
    train_targets_hybrid,
    minlength=5
)

hybrid_class_weights = (
    1.0 /
    hybrid_class_counts
)

hybrid_sample_weights = (
    hybrid_class_weights[
        train_targets_hybrid
    ]
)

hybrid_sample_weights = torch.as_tensor(
    hybrid_sample_weights,
    dtype=torch.double
)

hybrid_sampler = WeightedRandomSampler(
    weights=hybrid_sample_weights,
    num_samples=len(hybrid_sample_weights),
    replacement=True
)


# ============================================================
# 7. DataLoaders
# ============================================================

train_hybrid_loader = DataLoader(
    train_hybrid_dataset,
    batch_size=32,
    sampler=hybrid_sampler,
    num_workers=2,
    pin_memory=True
)

val_hybrid_loader = DataLoader(
    val_hybrid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# 8. Build ResNet18 from best checkpoint
# ============================================================

hybrid_resnet = models.resnet18(
    weights=None
)

hybrid_resnet.fc = nn.Linear(
    hybrid_resnet.fc.in_features,
    5
)

hybrid_resnet = hybrid_resnet.to(device)


# ============================================================
# 9. Load best CNN weights
# ============================================================

CHECKPOINT_PATH = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

checkpoint_hybrid = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

hybrid_resnet.load_state_dict(
    checkpoint_hybrid["model_state_dict"]
)


# ============================================================
# 10. Remove final classifier
#    Output = 512-dimensional deep feature
# ============================================================

deep_backbone = nn.Sequential(
    *list(hybrid_resnet.children())[:-1]
)

deep_backbone = deep_backbone.to(device)

deep_backbone.eval()


# ============================================================
# 11. Freeze CNN
# ============================================================

for param in deep_backbone.parameters():
    param.requires_grad = False


# ============================================================
# 12. Hybrid model
# ============================================================

class HybridNet(nn.Module):

    def __init__(
        self,
        deep_dim=512,
        handcrafted_dim=25,
        num_classes=5
    ):

        super().__init__()

        # --------------------------------------------
        # Handcrafted branch
        # --------------------------------------------

        self.handcrafted_branch = nn.Sequential(

            nn.Linear(
                handcrafted_dim,
                64
            ),

            nn.BatchNorm1d(64),

            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Linear(
                64,
                32
            ),

            nn.ReLU()
        )


        # --------------------------------------------
        # Fusion head
        # --------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                deep_dim + 32,
                128
            ),

            nn.BatchNorm1d(128),

            nn.ReLU(),

            nn.Dropout(0.30),

            nn.Linear(
                128,
                num_classes
            )
        )


    def forward(
        self,
        deep_features,
        handcrafted_features
    ):

        handcrafted_features = (
            self.handcrafted_branch(
                handcrafted_features
            )
        )

        fused = torch.cat(
            [
                deep_features,
                handcrafted_features
            ],
            dim=1
        )

        logits = self.classifier(
            fused
        )

        return logits


# ============================================================
# 13. Instantiate hybrid model
# ============================================================

hybrid_model = HybridNet(
    deep_dim=512,
    handcrafted_dim=25,
    num_classes=5
).to(device)


# ============================================================
# 14. Loss
# ============================================================

hybrid_criterion = nn.CrossEntropyLoss()


# ============================================================
# 15. Optimizer
#    CNN frozen: only fusion + handcrafted branch train
# ============================================================

hybrid_optimizer = torch.optim.Adam(
    hybrid_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


# ============================================================
# 16. Sanity check
# ============================================================

images, handcrafted, labels = next(
    iter(train_hybrid_loader)
)

images = images.to(
    device
)

handcrafted = handcrafted.to(
    device
)

with torch.no_grad():

    deep_features = deep_backbone(
        images
    )

deep_features = deep_features.flatten(
    start_dim=1
)

logits = hybrid_model(
    deep_features,
    handcrafted
)


# ============================================================
# 17. Final report
# ============================================================

print("=" * 70)
print("HYBRID NEURAL NETWORK READY")
print("=" * 70)

print(
    "CNN checkpoint Macro F1:",
    checkpoint_hybrid["macro_f1"]
)

print(
    "Deep feature dimension:",
    deep_features.shape[1]
)

print(
    "Handcrafted dimension:",
    handcrafted.shape[1]
)

print(
    "Fused dimension:",
    deep_features.shape[1]
    +
    handcrafted.shape[1]
)

print(
    "Output shape:",
    logits.shape
)

print(
    "CNN backbone frozen:",
    True
)

print(
    "Trainable Hybrid parameters:",
    sum(
        p.numel()
        for p in hybrid_model.parameters()
        if p.requires_grad
    )
)

print(
    "Sampler:",
    "WeightedRandomSampler"
)

print(
    "Loss:",
    "Unweighted CrossEntropy"
)

print(
    "Learning rate:",
    1e-4
)

print(
    "Device:",
    device
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 11 — HYBRID NEURAL NETWORK TRAINING
# ============================================================

from sklearn.metrics import f1_score


NUM_EPOCHS = 5

best_hybrid_f1 = -1.0
best_hybrid_epoch = 0

HYBRID_CHECKPOINT_PATH = (
    "/content/best_hybrid_seed2026.pth"
)


print("=" * 70)
print("HYBRID NEURAL NETWORK — TRAINING")
print("=" * 70)

print(
    "Starting CNN Macro F1:",
    checkpoint_hybrid["macro_f1"]
)

print(
    "Deep features: 512"
)

print(
    "Handcrafted branch: 25 -> 64 -> 32"
)

print(
    "Actual fusion dimension: 544"
)

print(
    "CNN frozen: True"
)

print(
    "Sampler: WeightedRandomSampler"
)

print(
    "Loss: Unweighted CrossEntropy"
)

print("=" * 70)


for epoch in range(NUM_EPOCHS):

    # ========================================================
    # TRAIN
    # ========================================================

    hybrid_model.train()

    # CNN is frozen and should remain in eval mode because
    # it contains BatchNorm layers.
    deep_backbone.eval()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, handcrafted, labels in train_hybrid_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        handcrafted = handcrafted.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        hybrid_optimizer.zero_grad()

        # ----------------------------------------------------
        # Deep features from frozen CNN
        # ----------------------------------------------------

        with torch.no_grad():

            deep_features = deep_backbone(
                images
            )

            deep_features = deep_features.flatten(
                start_dim=1
            )

        # ----------------------------------------------------
        # Hybrid prediction
        # ----------------------------------------------------

        outputs = hybrid_model(
            deep_features,
            handcrafted
        )

        loss = hybrid_criterion(
            outputs,
            labels
        )

        loss.backward()

        hybrid_optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)


    train_loss /= train_total

    train_acc = (
        train_correct /
        train_total
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    hybrid_model.eval()
    deep_backbone.eval()

    val_loss = 0.0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, handcrafted, labels in val_hybrid_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            handcrafted = handcrafted.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            # ------------------------------------------------
            # Frozen CNN
            # ------------------------------------------------

            deep_features = deep_backbone(
                images
            )

            deep_features = deep_features.flatten(
                start_dim=1
            )

            # ------------------------------------------------
            # Hybrid prediction
            # ------------------------------------------------

            outputs = hybrid_model(
                deep_features,
                handcrafted
            )

            loss = hybrid_criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_val_labels.extend(
                labels.cpu().numpy()
            )

            all_val_predictions.extend(
                predictions.cpu().numpy()
            )


    val_loss /= val_total

    val_acc = (
        sum(
            p == y
            for p, y in zip(
                all_val_predictions,
                all_val_labels
            )
        )
        /
        val_total
    )

    val_macro_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro",
        zero_division=0
    )


    # ========================================================
    # SAVE BEST HYBRID
    # ========================================================

    if val_macro_f1 > best_hybrid_f1:

        best_hybrid_f1 = val_macro_f1
        best_hybrid_epoch = epoch + 1

        torch.save(
            {
                "seed": SEED,
                "epoch": best_hybrid_epoch,
                "model_state_dict":
                    hybrid_model.state_dict(),
                "optimizer_state_dict":
                    hybrid_optimizer.state_dict(),
                "macro_f1":
                    val_macro_f1,
                "val_acc":
                    val_acc,
                "cnn_checkpoint_macro_f1":
                    checkpoint_hybrid["macro_f1"],
                "architecture":
                    "ResNet18 frozen + handcrafted MLP + fusion",
                "deep_dimension": 512,
                "handcrafted_dimension": 25,
                "handcrafted_branch_output": 32,
                "fusion_dimension": 544,
                "sampler":
                    "WeightedRandomSampler",
                "loss":
                    "UnweightedCrossEntropy"
            },
            HYBRID_CHECKPOINT_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""


    # ========================================================
    # PRINT
    # ========================================================

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {val_macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Starting CNN Macro F1: "
    f"{checkpoint_hybrid['macro_f1']:.4f}"
)

print(
    f"Best Hybrid Macro F1: "
    f"{best_hybrid_f1:.4f}"
)

print(
    f"Best epoch: "
    f"{best_hybrid_epoch}"
)

print(
    f"Checkpoint: "
    f"{HYBRID_CHECKPOINT_PATH}"
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 7 — KNEE-FOCUSED ROI PIPELINE
# ============================================================

import cv2
import numpy as np
import torch

from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler


# ------------------------------------------------------------
# 1. Center crop transform
# ------------------------------------------------------------

class CenterCropPercent:

    def __init__(self, ratio=0.70):
        if not (0 < ratio <= 1):
            raise ValueError(
                "ratio must be between 0 and 1"
            )

        self.ratio = ratio

    def __call__(self, img):

        width, height = img.size

        crop_width = int(
            width * self.ratio
        )

        crop_height = int(
            height * self.ratio
        )

        left = (
            width - crop_width
        ) // 2

        top = (
            height - crop_height
        ) // 2

        right = left + crop_width
        bottom = top + crop_height

        return img.crop(
            (
                left,
                top,
                right,
                bottom
            )
        )


# ------------------------------------------------------------
# 2. ROI training transform
# ------------------------------------------------------------

train_transform_roi = transforms.Compose([

    # NEW: focus on central knee region
    CenterCropPercent(
        ratio=0.70
    ),

    transforms.Resize(
        (224, 224)
    ),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    # Same augmentation as baseline
    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=5
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# 3. ROI evaluation transform
# ------------------------------------------------------------

eval_transform_roi = transforms.Compose([

    CenterCropPercent(
        ratio=0.70
    ),

    transforms.Resize(
        (224, 224)
    ),

    CLAHETransform(
        clip_limit=2.0,
        tile_grid_size=(8, 8)
    ),

    transforms.Grayscale(
        num_output_channels=3
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ------------------------------------------------------------
# 4. Create ROI datasets
# ------------------------------------------------------------

train_dataset_roi = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform_roi
)

val_dataset_roi = datasets.ImageFolder(
    VAL_DIR,
    transform=eval_transform_roi
)

test_dataset_roi = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform_roi
)


# ------------------------------------------------------------
# 5. Class distribution
# ------------------------------------------------------------

train_targets_roi = np.asarray(
    train_dataset_roi.targets
)

class_counts_roi = np.bincount(
    train_targets_roi,
    minlength=5
)


# ------------------------------------------------------------
# 6. WeightedRandomSampler
# ------------------------------------------------------------

sampling_weights_roi = (
    1.0 / class_counts_roi
)

sample_weights_roi = (
    sampling_weights_roi[
        train_targets_roi
    ]
)

sample_weights_roi = torch.as_tensor(
    sample_weights_roi,
    dtype=torch.double
)

sampler_roi = WeightedRandomSampler(
    weights=sample_weights_roi,
    num_samples=len(sample_weights_roi),
    replacement=True
)


# ------------------------------------------------------------
# 7. DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader_roi = DataLoader(
    train_dataset_roi,
    batch_size=BATCH_SIZE,
    sampler=sampler_roi,
    num_workers=2,
    pin_memory=True
)

val_loader_roi = DataLoader(
    val_dataset_roi,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader_roi = DataLoader(
    test_dataset_roi,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# 8. Sanity check
# ------------------------------------------------------------

images_roi, labels_roi = next(
    iter(train_loader_roi)
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print("=" * 70)
print("KNEE-FOCUSED ROI PIPELINE READY")
print("=" * 70)

print(
    "Crop ratio:",
    "70%"
)

print(
    "Train samples:",
    len(train_dataset_roi)
)

print(
    "Validation samples:",
    len(val_dataset_roi)
)

print(
    "Test samples:",
    len(test_dataset_roi)
)

print(
    "Batch shape:",
    images_roi.shape
)

print(
    "Class counts:",
    class_counts_roi.tolist()
)

print(
    "Sampler:",
    "WeightedRandomSampler"
)

print(
    "Input size:",
    "224x224"
)

print(
    "Device:",
    device
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("=" * 70)

In [ ]:
# ============================================================
# CELL 8 — ROI RESNET18 MODEL
# ============================================================

import torch.nn as nn
from torchvision import models


# ------------------------------------------------------------
# 1. Fresh ImageNet-pretrained ResNet18
# ------------------------------------------------------------

model_roi = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model_roi.fc = nn.Linear(
    model_roi.fc.in_features,
    5
)

model_roi = model_roi.to(device)


# ------------------------------------------------------------
# 2. Full fine-tuning
# ------------------------------------------------------------

for param in model_roi.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 3. Parameter count
# ------------------------------------------------------------

total_params_roi = sum(
    p.numel()
    for p in model_roi.parameters()
)

trainable_params_roi = sum(
    p.numel()
    for p in model_roi.parameters()
    if p.requires_grad
)


# ------------------------------------------------------------
# 4. Unweighted CrossEntropy
#    Same loss as winning Sampler strategy
# ------------------------------------------------------------

criterion_roi = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# 5. Optimizer
# ------------------------------------------------------------

optimizer_roi = torch.optim.Adam(
    model_roi.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# 6. Check
# ------------------------------------------------------------

print("=" * 70)
print("ROI RESNET18 MODEL READY")
print("=" * 70)

print(
    "Total parameters     :",
    f"{total_params_roi:,}"
)

print(
    "Trainable parameters :",
    f"{trainable_params_roi:,}"
)

print(
    "Input                : 70% center crop → 224x224"
)

print(
    "Pretrained           : ImageNet"
)

print(
    "Loss                 : Unweighted CrossEntropy"
)

print(
    "Sampler              : WeightedRandomSampler"
)

print(
    "Learning rate        : 1e-4"
)

print(
    "Backbone             : Fully trainable"
)

print(
    "Device               :",
    device
)

if torch.cuda.is_available():

    print(
        "GPU                  :",
        torch.cuda.get_device_name(0)
    )

print("=" * 70)

In [ ]:
# ============================================================
# CELL 9 — ROI RESNET18 TRAINING
# ============================================================

from sklearn.metrics import f1_score

NUM_EPOCHS = 5

best_macro_f1_roi = -1.0
best_epoch_roi = 0

BEST_ROI_PATH = (
    "/content/best_resnet18_roi_seed2026.pth"
)

print("=" * 70)
print("ROI RESNET18 — TRAINING")
print("=" * 70)

print("Baseline whole-image Macro F1 : 0.6691")
print("ROI crop                      : 70% center")
print("Sampler                       : WeightedRandomSampler")
print("Loss                          : Unweighted CrossEntropy")
print("Seed                          :", SEED)
print("=" * 70)


for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model_roi.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_roi:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_roi.zero_grad()

        outputs = model_roi(images)

        loss = criterion_roi(
            outputs,
            labels
        )

        loss.backward()

        optimizer_roi.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model_roi.eval()

    val_loss = 0.0
    val_total = 0

    all_labels_roi = []
    all_predictions_roi = []

    with torch.no_grad():

        for images, labels in val_loader_roi:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model_roi(images)

            loss = criterion_roi(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_labels_roi.extend(
                labels.cpu().numpy()
            )

            all_predictions_roi.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total

    val_acc = (
        sum(
            p == y
            for p, y in zip(
                all_predictions_roi,
                all_labels_roi
            )
        )
        / val_total
    )

    macro_f1 = f1_score(
        all_labels_roi,
        all_predictions_roi,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # SAVE BEST CHECKPOINT
    # --------------------------------------------------------

    if macro_f1 > best_macro_f1_roi:

        best_macro_f1_roi = macro_f1
        best_epoch_roi = epoch + 1

        torch.save(
            {
                "seed": SEED,
                "epoch": best_epoch_roi,
                "model_state_dict":
                    model_roi.state_dict(),
                "optimizer_state_dict":
                    optimizer_roi.state_dict(),
                "macro_f1": macro_f1,
                "val_acc": val_acc,
                "crop_ratio": 0.70,
                "resolution": "224x224",
                "sampler": "WeightedRandomSampler",
                "loss": "UnweightedCrossEntropy"
            },
            BEST_ROI_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""


    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best ROI Macro F1: "
    f"{best_macro_f1_roi:.4f}"
)

print(
    f"Best ROI epoch: "
    f"{best_epoch_roi}"
)

print(
    f"Checkpoint: "
    f"{BEST_ROI_PATH}"
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 10 — ORDINAL TARGETS
# ============================================================

import numpy as np
import torch


# ------------------------------------------------------------
# Convert KL grades to ordinal targets
# ------------------------------------------------------------

def create_ordinal_targets(labels):
    """
    Convert KL labels (0-4) into 4 cumulative
    binary threshold targets.

    Example:
        KL0 -> [0,0,0,0]
        KL1 -> [1,0,0,0]
        KL2 -> [1,1,0,0]
        KL3 -> [1,1,1,0]
        KL4 -> [1,1,1,1]
    """

    labels = np.asarray(
        labels,
        dtype=np.int64
    )

    ordinal_targets = np.zeros(
        (len(labels), 4),
        dtype=np.float32
    )

    for threshold in range(4):

        ordinal_targets[:, threshold] = (
            labels > threshold
        ).astype(np.float32)

    return ordinal_targets


# ------------------------------------------------------------
# Test the conversion
# ------------------------------------------------------------

test_labels = np.array([
    0, 1, 2, 3, 4
])

test_ordinal = create_ordinal_targets(
    test_labels
)


# ------------------------------------------------------------
# Display mapping
# ------------------------------------------------------------

print("=" * 70)
print("ORDINAL TARGET CONVERSION")
print("=" * 70)

for label, target in zip(
    test_labels,
    test_ordinal
):

    print(
        f"KL{label} -> "
        f"{target.astype(int).tolist()}"
    )


# ------------------------------------------------------------
# Create targets for our dataset
# ------------------------------------------------------------

train_labels_array = np.asarray(
    train_dataset.targets,
    dtype=np.int64
)

val_labels_array = np.asarray(
    val_dataset.targets,
    dtype=np.int64
)

test_labels_array = np.asarray(
    test_dataset.targets,
    dtype=np.int64
)

train_ordinal_targets = create_ordinal_targets(
    train_labels_array
)

val_ordinal_targets = create_ordinal_targets(
    val_labels_array
)

test_ordinal_targets = create_ordinal_targets(
    test_labels_array
)


# ------------------------------------------------------------
# Verify shapes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ORDINAL TARGETS READY")
print("=" * 70)

print(
    "Train labels:",
    train_labels_array.shape
)

print(
    "Train ordinal:",
    train_ordinal_targets.shape
)

print(
    "Validation labels:",
    val_labels_array.shape
)

print(
    "Validation ordinal:",
    val_ordinal_targets.shape
)

print(
    "Test labels:",
    test_labels_array.shape
)

print(
    "Test ordinal:",
    test_ordinal_targets.shape
)


# ------------------------------------------------------------
# Verify a few real samples
# ------------------------------------------------------------

print("\nFirst 10 training samples:")

for i in range(10):

    print(
        f"KL{train_labels_array[i]} -> "
        f"{train_ordinal_targets[i].astype(int).tolist()}"
    )

print("=" * 70)

In [ ]:
# ============================================================
# CELL 11 — ORDINAL RESNET18
# ============================================================

import torch
import torch.nn as nn
from torchvision import models


# ------------------------------------------------------------
# 1. Build ImageNet-pretrained ResNet18
# ------------------------------------------------------------

ordinal_model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)


# ------------------------------------------------------------
# 2. Replace 5-class head with 4 ordinal outputs
# ------------------------------------------------------------

ordinal_model.fc = nn.Linear(
    ordinal_model.fc.in_features,
    4
)

ordinal_model = ordinal_model.to(device)


# ------------------------------------------------------------
# 3. Full fine-tuning
# ------------------------------------------------------------

for param in ordinal_model.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 4. Binary CrossEntropy with logits
# ------------------------------------------------------------

ordinal_criterion = nn.BCEWithLogitsLoss()


# ------------------------------------------------------------
# 5. Optimizer
# ------------------------------------------------------------

ordinal_optimizer = torch.optim.Adam(
    ordinal_model.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# 6. Check
# ------------------------------------------------------------

total_params_ordinal = sum(
    p.numel()
    for p in ordinal_model.parameters()
)

trainable_params_ordinal = sum(
    p.numel()
    for p in ordinal_model.parameters()
    if p.requires_grad
)

print("=" * 70)
print("ORDINAL RESNET18 READY")
print("=" * 70)

print(
    "Architecture: ResNet18"
)

print(
    "Pretrained: ImageNet"
)

print(
    "Output dimension:",
    4
)

print(
    "Outputs represent:"
)

print(
    "  P(KL > 0)"
)

print(
    "  P(KL > 1)"
)

print(
    "  P(KL > 2)"
)

print(
    "  P(KL > 3)"
)

print(
    f"\nTotal parameters     : "
    f"{total_params_ordinal:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params_ordinal:,}"
)

print(
    "\nLoss:",
    "BCEWithLogitsLoss"
)

print(
    "Optimizer:",
    "Adam"
)

print(
    "Learning rate:",
    "1e-4"
)

print(
    "Sampler:",
    "WeightedRandomSampler"
)

print(
    "Input:",
    "224x224"
)

print(
    "Preprocessing:",
    "CLAHE + augmentation"
)

print(
    "Seed:",
    SEED
)

print(
    "Device:",
    device
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 12 — ORDINAL RESNET18 TRAINING
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    cohen_kappa_score
)


NUM_EPOCHS = 10

best_ordinal_f1 = -1.0
best_ordinal_epoch = 0

BEST_ORDINAL_PATH = (
    "/content/best_resnet18_ordinal_seed2026.pth"
)


# ------------------------------------------------------------
# Convert 4 threshold predictions -> KL grade
# ------------------------------------------------------------

def ordinal_logits_to_kl(
    logits
):
    """
    Convert four ordinal logits into a KL grade.

    Thresholds:
        output 0 -> KL > 0
        output 1 -> KL > 1
        output 2 -> KL > 2
        output 3 -> KL > 3

    The predicted KL grade is the number of
    positive thresholds.
    """

    probabilities = torch.sigmoid(
        logits
    )

    binary_predictions = (
        probabilities >= 0.5
    ).int()

    predicted_kl = (
        binary_predictions.sum(
            dim=1
        )
    )

    return predicted_kl


print("=" * 70)
print("ORDINAL RESNET18 — TRAINING")
print("=" * 70)

print(
    "Baseline CNN Macro F1 : 0.6691"
)

print(
    "Baseline CNN QWK      : 0.7980"
)

print(
    "Ordinal thresholds    : 4"
)

print(
    "Loss                  : BCEWithLogitsLoss"
)

print(
    "Sampler               : WeightedRandomSampler"
)

print(
    "Seed                  :",
    SEED
)

print("=" * 70)


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(NUM_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    ordinal_model.train()

    train_loss = 0.0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        # Convert KL labels -> ordinal targets
        ordinal_targets = torch.zeros(
            labels.size(0),
            4,
            dtype=torch.float32,
            device=device
        )

        for threshold in range(4):

            ordinal_targets[:, threshold] = (
                labels > threshold
            ).float()

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        ordinal_optimizer.zero_grad()

        logits = ordinal_model(
            images
        )

        loss = ordinal_criterion(
            logits,
            ordinal_targets
        )

        loss.backward()

        ordinal_optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        train_loss += (
            loss.item()
            * images.size(0)
        )

        train_total += labels.size(0)


    train_loss /= train_total


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    ordinal_model.eval()

    val_loss = 0.0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            # ------------------------------------------------
            # Ordinal targets
            # ------------------------------------------------

            ordinal_targets = torch.zeros(
                labels.size(0),
                4,
                dtype=torch.float32,
                device=device
            )

            for threshold in range(4):

                ordinal_targets[:, threshold] = (
                    labels > threshold
                ).float()

            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            logits = ordinal_model(
                images
            )

            loss = ordinal_criterion(
                logits,
                ordinal_targets
            )

            val_loss += (
                loss.item()
                * images.size(0)
            )

            val_total += labels.size(0)

            # ------------------------------------------------
            # Convert ordinal outputs -> KL
            # ------------------------------------------------

            predicted_kl = (
                ordinal_logits_to_kl(
                    logits
                )
            )

            all_val_labels.extend(
                labels.cpu().numpy()
            )

            all_val_predictions.extend(
                predicted_kl.cpu().numpy()
            )


    val_loss /= val_total


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    val_accuracy = accuracy_score(
        all_val_labels,
        all_val_predictions
    )

    val_balanced_accuracy = (
        balanced_accuracy_score(
            all_val_labels,
            all_val_predictions
        )
    )

    val_macro_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro",
        zero_division=0
    )

    val_mae = mean_absolute_error(
        all_val_labels,
        all_val_predictions
    )

    val_qwk = cohen_kappa_score(
        all_val_labels,
        all_val_predictions,
        weights="quadratic"
    )


    # --------------------------------------------------------
    # Save best checkpoint
    # --------------------------------------------------------

    if val_macro_f1 > best_ordinal_f1:

        best_ordinal_f1 = val_macro_f1
        best_ordinal_epoch = epoch + 1

        torch.save(
            {
                "seed": SEED,
                "epoch": best_ordinal_epoch,
                "model_state_dict":
                    ordinal_model.state_dict(),
                "optimizer_state_dict":
                    ordinal_optimizer.state_dict(),
                "macro_f1":
                    val_macro_f1,
                "val_acc":
                    val_accuracy,
                "balanced_accuracy":
                    val_balanced_accuracy,
                "mae":
                    val_mae,
                "qwk":
                    val_qwk,
                "architecture":
                    "ResNet18 ordinal",
                "num_thresholds":
                    4,
                "loss":
                    "BCEWithLogitsLoss",
                "sampler":
                    "WeightedRandomSampler",
                "resolution":
                    "224x224"
            },
            BEST_ORDINAL_PATH
        )

        saved = " <-- NEW BEST"

    else:

        saved = ""


    # --------------------------------------------------------
    # Print epoch
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Balanced Acc: {val_balanced_accuracy:.4f} | "
        f"Macro F1: {val_macro_f1:.4f} | "
        f"QWK: {val_qwk:.4f} | "
        f"MAE: {val_mae:.4f}"
        f"{saved}"
    )


print("-" * 70)

print(
    f"Best Ordinal Macro F1: "
    f"{best_ordinal_f1:.4f}"
)

print(
    f"Best epoch: "
    f"{best_ordinal_epoch}"
)

print(
    f"Checkpoint: "
    f"{BEST_ORDINAL_PATH}"
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 13 — EARLY-STAGE DATASET
# KL0 vs KL1 vs KL2
# ============================================================

import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)


# ------------------------------------------------------------
# Early-stage labels
# ------------------------------------------------------------

EARLY_CLASSES = [0, 1, 2]

EARLY_CLASS_NAMES = {
    0: "Normal",
    1: "Doubtful",
    2: "Mild"
}


# ------------------------------------------------------------
# Custom subset that keeps only KL0, KL1, KL2
# ------------------------------------------------------------

class EarlyDataset(Dataset):

    def __init__(
        self,
        base_dataset
    ):

        self.base_dataset = base_dataset

        self.indices = [
            idx
            for idx, label
            in enumerate(base_dataset.targets)
            if label in EARLY_CLASSES
        ]

        self.labels = np.array(
            [
                base_dataset.targets[idx]
                for idx in self.indices
            ],
            dtype=np.int64
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        original_idx = self.indices[idx]

        image, original_label = (
            self.base_dataset[original_idx]
        )

        # Labels remain 0,1,2
        return (
            image,
            torch.tensor(
                original_label,
                dtype=torch.long
            )
        )


# ------------------------------------------------------------
# Create Early train/validation datasets
# ------------------------------------------------------------

early_train_dataset = EarlyDataset(
    train_dataset
)

early_val_dataset = EarlyDataset(
    val_dataset
)


# ------------------------------------------------------------
# Class distribution
# ------------------------------------------------------------

early_train_counts = np.bincount(
    early_train_dataset.labels,
    minlength=3
)

early_val_counts = np.bincount(
    early_val_dataset.labels,
    minlength=3
)


# ------------------------------------------------------------
# WeightedRandomSampler
# ------------------------------------------------------------

early_sampling_weights = (
    1.0 /
    early_train_counts
)

early_sample_weights = (
    early_sampling_weights[
        early_train_dataset.labels
    ]
)

early_sample_weights = torch.as_tensor(
    early_sample_weights,
    dtype=torch.double
)

early_sampler = WeightedRandomSampler(
    weights=early_sample_weights,
    num_samples=len(
        early_sample_weights
    ),
    replacement=True
)


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

early_train_loader = DataLoader(
    early_train_dataset,
    batch_size=32,
    sampler=early_sampler,
    num_workers=2,
    pin_memory=True
)

early_val_loader = DataLoader(
    early_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

early_images, early_labels = next(
    iter(early_train_loader)
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 70)
print("EARLY-STAGE DATASET READY")
print("=" * 70)

print(
    "Train samples:",
    len(early_train_dataset)
)

print(
    "Validation samples:",
    len(early_val_dataset)
)

print("\nTraining class counts:")

for cls in EARLY_CLASSES:

    print(
        f"KL{cls} "
        f"({EARLY_CLASS_NAMES[cls]}): "
        f"{early_train_counts[cls]}"
    )

print("\nValidation class counts:")

for cls in EARLY_CLASSES:

    print(
        f"KL{cls} "
        f"({EARLY_CLASS_NAMES[cls]}): "
        f"{early_val_counts[cls]}"
    )

print("\nBatch shape:")
print(early_images.shape)

print(
    "\nNumber of train batches:",
    len(early_train_loader)
)

print(
    "Number of validation batches:",
    len(early_val_loader)
)

print(
    "\nSampler:",
    "WeightedRandomSampler"
)

print(
    "Number of classes:",
    3
)

print(
    "Device:",
    device
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print("=" * 70)

### Quantitative Feature Space Analysis
To test hypothesis 29 (Representation Limitation), we will calculate Silhouette Scores and Inter-class distances in the original 512D space instead of relying on 2D t-SNE.

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples
import pandas as pd
import seaborn as sns

# Using 'all_features' and 'all_labels' extracted in Cell 19
# Calculate global silhouette score for KL0, KL1, KL2
mask_012 = np.isin(all_labels, [0, 1, 2])
features_012 = all_features[mask_012]
labels_012 = all_labels[mask_012]

global_score = silhouette_score(features_012, labels_012)
sample_scores = silhouette_samples(features_012, labels_012)

print(f"Global Silhouette Score (KL0/1/2): {global_score:.4f}")

# Per-class average silhouette score
sil_df = pd.DataFrame({'label': labels_012, 'silhouette': sample_scores})
class_avg = sil_df.groupby('label')['silhouette'].mean()
print("\nAverage Silhouette Score per class:")
print(class_avg)

# Interpreting: Scores near 0 indicate significant overlap/ambiguity.

In [ ]:
# ============================================================
# CELL 14 — EARLY-STAGE RESNET18
# KL0 vs KL1 vs KL2
# ============================================================

import torch.nn as nn
from torchvision import models


# ------------------------------------------------------------
# 1. Fresh ImageNet-pretrained ResNet18
# ------------------------------------------------------------

early_model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

early_model.fc = nn.Linear(
    early_model.fc.in_features,
    3
)

early_model = early_model.to(device)


# ------------------------------------------------------------
# 2. Full fine-tuning
# ------------------------------------------------------------

for param in early_model.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 3. Loss
# ------------------------------------------------------------

early_criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# 4. Optimizer
# ------------------------------------------------------------

early_optimizer = torch.optim.Adam(
    early_model.parameters(),
    lr=1e-4
)


# ------------------------------------------------------------
# 5. Parameter check
# ------------------------------------------------------------

early_total_params = sum(
    p.numel()
    for p in early_model.parameters()
)

early_trainable_params = sum(
    p.numel()
    for p in early_model.parameters()
    if p.requires_grad
)


# ------------------------------------------------------------
# 6. Sanity check
# ------------------------------------------------------------

test_images, test_labels = next(
    iter(early_train_loader)
)

test_images = test_images.to(device)

with torch.no_grad():
    test_outputs = early_model(test_images)


# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print("=" * 70)
print("EARLY-STAGE RESNET18 READY")
print("=" * 70)

print(
    "Classes:",
    "KL0, KL1, KL2"
)

print(
    "Output classes:",
    3
)

print(
    "Total parameters:",
    f"{early_total_params:,}"
)

print(
    "Trainable parameters:",
    f"{early_trainable_params:,}"
)

print(
    "Input:",
    "224x224"
)

print(
    "Preprocessing:",
    "CLAHE + augmentation"
)

print(
    "Sampler:",
    "WeightedRandomSampler"
)

print(
    "Loss:",
    "Unweighted CrossEntropy"
)

print(
    "Learning rate:",
    "1e-4"
)

print(
    "Seed:",
    SEED
)

print(
    "Output batch shape:",
    test_outputs.shape
)

print(
    "Device:",
    device
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 15 — EARLY-STAGE TRAINING
# KL0 vs KL1 vs KL2
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


NUM_EPOCHS = 10

best_early_f1 = -1.0
best_early_epoch = 0
best_early_kl1_f1 = 0.0

BEST_EARLY_PATH = (
    "/content/best_resnet18_early_seed2026.pth"
)


print("=" * 70)
print("EARLY-STAGE RESNET18 — TRAINING")
print("=" * 70)

print("Task: KL0 vs KL1 vs KL2")
print("Baseline full 5-class KL1 F1: 0.3023")
print("Baseline full 5-class Macro F1: 0.6691")
print("Sampler: WeightedRandomSampler")
print("Loss: Unweighted CrossEntropy")
print("Seed:", SEED)
print("=" * 70)


for epoch in range(NUM_EPOCHS):

    # ========================================================
    # TRAIN
    # ========================================================

    early_model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in early_train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        early_optimizer.zero_grad()

        outputs = early_model(
            images
        )

        loss = early_criterion(
            outputs,
            labels
        )

        loss.backward()

        early_optimizer.step()

        train_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total

    train_acc = (
        train_correct /
        train_total
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    early_model.eval()

    val_loss = 0.0
    val_total = 0

    all_early_labels = []
    all_early_predictions = []

    with torch.no_grad():

        for images, labels in early_val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = early_model(
                images
            )

            loss = early_criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item()
                * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            all_early_labels.extend(
                labels.cpu().numpy()
            )

            all_early_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= val_total


    # ========================================================
    # METRICS
    # ========================================================

    val_accuracy = accuracy_score(
        all_early_labels,
        all_early_predictions
    )

    val_balanced_accuracy = (
        balanced_accuracy_score(
            all_early_labels,
            all_early_predictions
        )
    )

    val_macro_f1 = f1_score(
        all_early_labels,
        all_early_predictions,
        average="macro",
        zero_division=0
    )

    per_class_f1 = f1_score(
        all_early_labels,
        all_early_predictions,
        average=None,
        labels=[0, 1, 2],
        zero_division=0
    )

    kl1_f1 = per_class_f1[1]


    # ========================================================
    # SAVE BEST BY MACRO F1
    # ========================================================

    if val_macro_f1 > best_early_f1:

        best_early_f1 = val_macro_f1
        best_early_epoch = epoch + 1
        best_early_kl1_f1 = kl1_f1

        torch.save(
            {
                "seed": SEED,
                "epoch": best_early_epoch,
                "model_state_dict":
                    early_model.state_dict(),
                "optimizer_state_dict":
                    early_optimizer.state_dict(),
                "macro_f1":
                    val_macro_f1,
                "val_acc":
                    val_accuracy,
                "balanced_accuracy":
                    val_balanced_accuracy,
                "kl1_f1":
                    kl1_f1,
                "task":
                    "KL0_vs_KL1_vs_KL2",
                "sampler":
                    "WeightedRandomSampler",
                "loss":
                    "UnweightedCrossEntropy",
                "resolution":
                    "224x224"
            },
            BEST_EARLY_PATH
        )

        saved = " <-- NEW BEST"

    else:
        saved = ""


    # ========================================================
    # EPOCH REPORT
    # ========================================================

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Balanced Acc: {val_balanced_accuracy:.4f} | "
        f"Macro F1: {val_macro_f1:.4f} | "
        f"KL1 F1: {kl1_f1:.4f}"
        f"{saved}"
    )


# ============================================================
# FINAL REPORT
# ============================================================

print("-" * 70)

print(
    f"Best Early Macro F1: "
    f"{best_early_f1:.4f}"
)

print(
    f"Best epoch: "
    f"{best_early_epoch}"
)

print(
    f"Best KL1 F1: "
    f"{best_early_kl1_f1:.4f}"
)

print(
    f"Checkpoint: "
    f"{BEST_EARLY_PATH}"
)

print("=" * 70)


# ============================================================
# LOAD BEST MODEL FOR FINAL EARLY REPORT
# ============================================================

best_early_checkpoint = torch.load(
    BEST_EARLY_PATH,
    map_location=device,
    weights_only=False
)

early_model.load_state_dict(
    best_early_checkpoint["model_state_dict"]
)

early_model.eval()


# ============================================================
# FINAL VALIDATION PREDICTIONS
# ============================================================

final_labels = []
final_predictions = []

with torch.no_grad():

    for images, labels in early_val_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        outputs = early_model(
            images
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        final_labels.extend(
            labels.numpy()
        )

        final_predictions.extend(
            predictions.cpu().numpy()
        )


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("=" * 70)
print("BEST EARLY MODEL — CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        final_labels,
        final_predictions,
        labels=[0, 1, 2],
        target_names=[
            "Normal",
            "Doubtful",
            "Mild"
        ],
        digits=4,
        zero_division=0
    )
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

print("=" * 70)
print("BEST EARLY MODEL — CONFUSION MATRIX")
print("=" * 70)

print(
    confusion_matrix(
        final_labels,
        final_predictions,
        labels=[0, 1, 2]
    )
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 16 — CASCADE INFERENCE
# Baseline 5-class -> Early-only refinement
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn

from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    cohen_kappa_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. Paths
# ============================================================

BASELINE_CHECKPOINT = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

EARLY_CHECKPOINT = (
    "/content/best_resnet18_early_seed2026.pth"
)


if not os.path.exists(BASELINE_CHECKPOINT):
    raise FileNotFoundError(
        f"Baseline checkpoint not found:\n"
        f"{BASELINE_CHECKPOINT}"
    )

if not os.path.exists(EARLY_CHECKPOINT):
    raise FileNotFoundError(
        f"Early checkpoint not found:\n"
        f"{EARLY_CHECKPOINT}"
    )


# ============================================================
# 2. Rebuild baseline model
# ============================================================

baseline_model = models.resnet18(
    weights=None
)

baseline_model.fc = nn.Linear(
    baseline_model.fc.in_features,
    5
)

baseline_model = baseline_model.to(device)

baseline_checkpoint = torch.load(
    BASELINE_CHECKPOINT,
    map_location=device,
    weights_only=False
)

baseline_model.load_state_dict(
    baseline_checkpoint["model_state_dict"]
)

baseline_model.eval()


# ============================================================
# 3. Rebuild Early-only model
# ============================================================

early_model_cascade = models.resnet18(
    weights=None
)

early_model_cascade.fc = nn.Linear(
    early_model_cascade.fc.in_features,
    3
)

early_model_cascade = early_model_cascade.to(device)

early_checkpoint_cascade = torch.load(
    EARLY_CHECKPOINT,
    map_location=device,
    weights_only=False
)

early_model_cascade.load_state_dict(
    early_checkpoint_cascade["model_state_dict"]
)

early_model_cascade.eval()


# ============================================================
# 4. Validation inference
# ============================================================

baseline_predictions = []
cascade_predictions = []
true_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        # ---------------------------------------------
        # Baseline prediction
        # ---------------------------------------------

        baseline_logits = baseline_model(
            images
        )

        baseline_pred = torch.argmax(
            baseline_logits,
            dim=1
        )


        # ---------------------------------------------
        # Early-only prediction
        # ---------------------------------------------

        early_logits = early_model_cascade(
            images
        )

        early_pred = torch.argmax(
            early_logits,
            dim=1
        )


        # ---------------------------------------------
        # Cascade
        # ---------------------------------------------

        cascade_pred = baseline_pred.clone()

        early_mask = (
            baseline_pred <= 2
        )

        cascade_pred[
            early_mask
        ] = early_pred[
            early_mask
        ]


        # ---------------------------------------------
        # Store
        # ---------------------------------------------

        baseline_predictions.extend(
            baseline_pred.cpu().numpy()
        )

        cascade_predictions.extend(
            cascade_pred.cpu().numpy()
        )

        true_labels.extend(
            labels.numpy()
        )


# ============================================================
# 5. Evaluation function
# ============================================================

def evaluate_predictions(
    y_true,
    y_pred,
    model_name
):

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_acc = balanced_accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,
        weights="quadratic"
    )

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    print(
        f"Accuracy          : {accuracy:.4f}"
    )

    print(
        f"Balanced Accuracy : {balanced_acc:.4f}"
    )

    print(
        f"Macro F1          : {macro_f1:.4f}"
    )

    print(
        f"MAE               : {mae:.4f}"
    )

    print(
        f"QWK               : {qwk:.4f}"
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_acc,
        "Macro F1": macro_f1,
        "MAE": mae,
        "QWK": qwk
    }


# ============================================================
# 6. Baseline vs Cascade
# ============================================================

baseline_results = evaluate_predictions(
    true_labels,
    baseline_predictions,
    "BASELINE 5-CLASS"
)

cascade_results = evaluate_predictions(
    true_labels,
    cascade_predictions,
    "CASCADE"
)


# ============================================================
# 7. Per-class reports
# ============================================================

class_names_5 = [
    "Normal",
    "Doubtful",
    "Mild",
    "Moderate",
    "Severe"
]

print("\n" + "=" * 70)
print("BASELINE CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        true_labels,
        baseline_predictions,
        labels=[0, 1, 2, 3, 4],
        target_names=class_names_5,
        digits=4,
        zero_division=0
    )
)


print("=" * 70)
print("CASCADE CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        true_labels,
        cascade_predictions,
        labels=[0, 1, 2, 3, 4],
        target_names=class_names_5,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 8. Confusion matrices
# ============================================================

baseline_cm = confusion_matrix(
    true_labels,
    baseline_predictions,
    labels=[0, 1, 2, 3, 4]
)

cascade_cm = confusion_matrix(
    true_labels,
    cascade_predictions,
    labels=[0, 1, 2, 3, 4]
)


print("=" * 70)
print("BASELINE CONFUSION MATRIX")
print("=" * 70)

print(baseline_cm)


print("\n" + "=" * 70)
print("CASCADE CONFUSION MATRIX")
print("=" * 70)

print(cascade_cm)


# ============================================================
# 9. Compare only Early classes
# ============================================================

early_true_mask = np.isin(
    np.asarray(true_labels),
    [0, 1, 2]
)

early_true = np.asarray(
    true_labels
)[early_true_mask]

early_baseline_pred = np.asarray(
    baseline_predictions
)[early_true_mask]

early_cascade_pred = np.asarray(
    cascade_predictions
)[early_true_mask]


print("\n" + "=" * 70)
print("EARLY-REGION COMPARISON")
print("=" * 70)

print(
    "Baseline Early Macro F1:",
    f"{f1_score(early_true, early_baseline_pred, average='macro', zero_division=0):.4f}"
)

print(
    "Cascade Early Macro F1 :",
    f"{f1_score(early_true, early_cascade_pred, average='macro', zero_division=0):.4f}"
)

print(
    "Baseline KL1 F1:",
    f"{f1_score(early_true, early_baseline_pred, labels=[1], average='macro', zero_division=0):.4f}"
)

print(
    "Cascade KL1 F1:",
    f"{f1_score(early_true, early_cascade_pred, labels=[1], average='macro', zero_division=0):.4f}"
)


# ============================================================
# 10. Final comparison
# ============================================================

print("\n" + "=" * 70)
print("FINAL CASCADE COMPARISON")
print("=" * 70)

print(
    f"Baseline Macro F1 : "
    f"{baseline_results['Macro F1']:.4f}"
)

print(
    f"Cascade Macro F1  : "
    f"{cascade_results['Macro F1']:.4f}"
)

print(
    f"Difference         : "
    f"{cascade_results['Macro F1'] - baseline_results['Macro F1']:+.4f}"
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 17 — GRAD-CAM DIAGNOSTIC
# Baseline vs Early-only
# Focus: KL1 correct / KL1->KL0 / KL1->KL2
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import models


# ============================================================
# 1. CHECKPOINTS
# ============================================================

BASELINE_CHECKPOINT = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

EARLY_CHECKPOINT = (
    "/content/best_resnet18_early_seed2026.pth"
)

if not os.path.exists(BASELINE_CHECKPOINT):
    raise FileNotFoundError(
        f"Baseline checkpoint not found:\n"
        f"{BASELINE_CHECKPOINT}"
    )

if not os.path.exists(EARLY_CHECKPOINT):
    raise FileNotFoundError(
        f"Early checkpoint not found:\n"
        f"{EARLY_CHECKPOINT}"
    )


# ============================================================
# 2. REBUILD BASELINE MODEL
# ============================================================

baseline_gradcam_model = models.resnet18(
    weights=None
)

baseline_gradcam_model.fc = nn.Linear(
    baseline_gradcam_model.fc.in_features,
    5
)

baseline_gradcam_model.load_state_dict(
    torch.load(
        BASELINE_CHECKPOINT,
        map_location=device,
        weights_only=False
    )["model_state_dict"]
)

baseline_gradcam_model = (
    baseline_gradcam_model.to(device)
)

baseline_gradcam_model.eval()


# ============================================================
# 3. REBUILD EARLY-ONLY MODEL
# ============================================================

early_gradcam_model = models.resnet18(
    weights=None
)

early_gradcam_model.fc = nn.Linear(
    early_gradcam_model.fc.in_features,
    3
)

early_gradcam_model.load_state_dict(
    torch.load(
        EARLY_CHECKPOINT,
        map_location=device,
        weights_only=False
    )["model_state_dict"]
)

early_gradcam_model = (
    early_gradcam_model.to(device)
)

early_gradcam_model.eval()


# ============================================================
# 4. Grad-CAM helper
# ============================================================

class GradCAM:

    def __init__(
        self,
        model,
        target_layer
    ):

        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_handle = (
            target_layer.register_forward_hook(
                self.save_activations
            )
        )

        self.backward_handle = (
            target_layer.register_full_backward_hook(
                self.save_gradients
            )
        )

    def save_activations(
        self,
        module,
        input,
        output
    ):

        self.activations = output

    def save_gradients(
        self,
        module,
        grad_input,
        grad_output
    ):

        self.gradients = grad_output[0]

    def generate(
        self,
        image,
        target_class
    ):

        self.model.zero_grad(
            set_to_none=True
        )

        output = self.model(
            image
        )

        score = output[
            0,
            target_class
        ]

        score.backward()

        activations = (
            self.activations[0]
        )

        gradients = (
            self.gradients[0]
        )

        # Global average pooling over spatial dimensions
        weights = gradients.mean(
            dim=(1, 2)
        )

        cam = (
            weights[:, None, None]
            * activations
        ).sum(
            dim=0
        )

        cam = torch.relu(
            cam
        )

        cam = cam.detach().cpu().numpy()

        # Normalize to [0,1]
        cam_min = cam.min()
        cam_max = cam.max()

        if cam_max > cam_min:

            cam = (
                cam - cam_min
            ) / (
                cam_max - cam_min
            )

        else:

            cam = np.zeros_like(
                cam
            )

        return output.detach(), cam

    def close(self):

        self.forward_handle.remove()
        self.backward_handle.remove()


# ============================================================
# 5. Create Grad-CAM objects
# ============================================================

baseline_cam = GradCAM(
    baseline_gradcam_model,
    baseline_gradcam_model.layer4[-1]
)

early_cam = GradCAM(
    early_gradcam_model,
    early_gradcam_model.layer4[-1]
)


# ============================================================
# 6. Collect KL1 validation examples
# ============================================================

# We use the existing validation dataset.
# No training augmentation is used here.

KL1_CASES = {
    "correct": [],
    "to_KL0": [],
    "to_KL2": []
}

MAX_PER_GROUP = 2

for idx in range(
    len(val_dataset)
):

    if all(
        len(v) >= MAX_PER_GROUP
        for v in KL1_CASES.values()
    ):
        break

    # Original validation image and label
    image_tensor, label = (
        val_dataset[idx]
    )

    if label != 1:
        continue

    image_batch = (
        image_tensor
        .unsqueeze(0)
        .to(device)
    )

    # Baseline prediction
    with torch.no_grad():

        baseline_output = (
            baseline_gradcam_model(
                image_batch
            )
        )

        baseline_pred = int(
            torch.argmax(
                baseline_output,
                dim=1
            ).item()
        )

    if baseline_pred == 1:

        group = "correct"

    elif baseline_pred == 0:

        group = "to_KL0"

    elif baseline_pred == 2:

        group = "to_KL2"

    else:

        continue

    if len(KL1_CASES[group]) < MAX_PER_GROUP:

        KL1_CASES[group].append(
            {
                "index": idx,
                "image": image_tensor,
                "baseline_pred": baseline_pred
            }
        )


# ============================================================
# 7. Print available cases
# ============================================================

print("=" * 70)
print("KL1 CASES FOUND")
print("=" * 70)

for group, cases in KL1_CASES.items():

    print(
        f"{group}: {len(cases)}"
    )

    for case in cases:

        print(
            f"  index={case['index']} "
            f"| baseline prediction=KL"
            f"{case['baseline_pred']}"
        )

print("=" * 70)


# ============================================================
# 8. Visualization helper
# ============================================================

def tensor_to_gray_image(
    tensor
):

    image = tensor.detach().cpu().numpy()

    # [3,H,W] -> [H,W]
    image = image.mean(
        axis=0
    )

    # Since normalized, rescale only for display
    image = (
        image - image.min()
    ) / (
        image.max() - image.min() + 1e-8
    )

    return image


def show_case(
    case,
    group_name
):

    image_tensor = (
        case["image"]
    )

    image_batch = (
        image_tensor
        .unsqueeze(0)
        .to(device)
    )

    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    baseline_output, baseline_heatmap = (
        baseline_cam.generate(
            image_batch,
            case["baseline_pred"]
        )
    )

    baseline_pred = int(
        torch.argmax(
            baseline_output,
            dim=1
        ).item()
    )


    # --------------------------------------------------------
    # Early-only
    # --------------------------------------------------------

    with torch.no_grad():

        early_output = (
            early_gradcam_model(
                image_batch
            )
        )

        early_pred = int(
            torch.argmax(
                early_output,
                dim=1
            ).item()
        )

    _, early_heatmap = (
        early_cam.generate(
            image_batch,
            early_pred
        )
    )


    # --------------------------------------------------------
    # Original image
    # --------------------------------------------------------

    gray_image = tensor_to_gray_image(
        image_tensor
    )


    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(16, 4)
    )

    # Original
    axes[0].imshow(
        gray_image,
        cmap="gray"
    )

    axes[0].set_title(
        f"Original\nTrue KL1"
    )

    axes[0].axis("off")


    # Baseline
    axes[1].imshow(
        gray_image,
        cmap="gray"
    )

    axes[1].imshow(
        baseline_heatmap,
        cmap="jet",
        alpha=0.45
    )

    axes[1].set_title(
        f"Baseline\nPred KL{baseline_pred}"
    )

    axes[1].axis("off")


    # Early-only
    axes[2].imshow(
        gray_image,
        cmap="gray"
    )

    axes[2].imshow(
        early_heatmap,
        cmap="jet",
        alpha=0.45
    )

    axes[2].set_title(
        f"Early-only\nPred KL{early_pred}"
    )

    axes[2].axis("off")


    # Side-by-side heatmap difference
    axes[3].imshow(
        baseline_heatmap,
        cmap="jet"
    )

    axes[3].set_title(
        "Baseline CAM"
    )

    axes[3].axis("off")

    plt.suptitle(
        f"KL1 case: {group_name} | "
        f"Index {case['index']}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# 9. Show all selected cases
# ============================================================

for group_name, cases in KL1_CASES.items():

    for case in cases:

        show_case(
            case,
            group_name
        )


# ============================================================
# 10. Cleanup hooks
# ============================================================

baseline_cam.close()
early_cam.close()

print("=" * 70)
print("GRAD-CAM ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 18 — QUANTITATIVE GRAD-CAM ANALYSIS
# All KL1 validation samples
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision import models
from sklearn.metrics import f1_score


# ============================================================
# 1. Checkpoint
# ============================================================

BASELINE_CHECKPOINT = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

if not os.path.exists(BASELINE_CHECKPOINT):

    raise FileNotFoundError(
        f"Baseline checkpoint not found:\n"
        f"{BASELINE_CHECKPOINT}"
    )


# ============================================================
# 2. Build baseline model
# ============================================================

gradcam_model = models.resnet18(
    weights=None
)

gradcam_model.fc = nn.Linear(
    gradcam_model.fc.in_features,
    5
)

checkpoint_gc = torch.load(
    BASELINE_CHECKPOINT,
    map_location=device,
    weights_only=False
)

gradcam_model.load_state_dict(
    checkpoint_gc["model_state_dict"]
)

gradcam_model = gradcam_model.to(
    device
)

gradcam_model.eval()


# ============================================================
# 3. Grad-CAM class
# ============================================================

class QuantitativeGradCAM:

    def __init__(
        self,
        model,
        target_layer
    ):

        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_handle = (
            target_layer.register_forward_hook(
                self._save_activations
            )
        )

        self.backward_handle = (
            target_layer.register_full_backward_hook(
                self._save_gradients
            )
        )

    def _save_activations(
        self,
        module,
        inputs,
        output
    ):

        self.activations = output

    def _save_gradients(
        self,
        module,
        grad_input,
        grad_output
    ):

        self.gradients = grad_output[0]

    def generate(
        self,
        image,
        target_class
    ):

        self.model.zero_grad(
            set_to_none=True
        )

        output = self.model(
            image
        )

        score = output[
            0,
            target_class
        ]

        score.backward()

        activations = (
            self.activations[0]
        )

        gradients = (
            self.gradients[0]
        )

        # Channel weights
        weights = gradients.mean(
            dim=(1, 2)
        )

        # Weighted activation map
        cam = (
            weights[:, None, None]
            * activations
        ).sum(
            dim=0
        )

        cam = torch.relu(
            cam
        )

        cam = cam.detach().cpu().numpy()

        # Normalize to [0,1]
        cam_min = cam.min()
        cam_max = cam.max()

        if cam_max > cam_min:

            cam = (
                cam - cam_min
            ) / (
                cam_max - cam_min
            )

        else:

            cam = np.zeros_like(
                cam
            )

        return output.detach(), cam

    def close(self):

        self.forward_handle.remove()
        self.backward_handle.remove()


# ============================================================
# 4. Create Grad-CAM
# ============================================================

cam_generator = QuantitativeGradCAM(
    gradcam_model,
    gradcam_model.layer4[-1]
)


# ============================================================
# 5. Quantitative CAM metrics
# ============================================================

def cam_metrics(cam):

    # Flatten
    values = cam.flatten().astype(
        np.float64
    )

    # Numerical stability
    values = values + 1e-12

    # Convert to probability distribution
    probabilities = (
        values /
        values.sum()
    )

    # --------------------------------------------------------
    # Entropy
    # --------------------------------------------------------

    entropy = -np.sum(
        probabilities *
        np.log(
            probabilities
        )
    )

    # Normalize entropy to [0,1]
    max_entropy = np.log(
        len(probabilities)
    )

    normalized_entropy = (
        entropy /
        max_entropy
        if max_entropy > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Top-k concentration
    # --------------------------------------------------------

    sorted_values = np.sort(
        probabilities
    )[::-1]

    top1_concentration = (
        sorted_values[:1].sum()
    )

    top3_concentration = (
        sorted_values[:3].sum()
    )

    top5_concentration = (
        sorted_values[:5].sum()
    )

    # --------------------------------------------------------
    # Spatial spread
    # --------------------------------------------------------

    h, w = cam.shape

    yy, xx = np.mgrid[
        0:h,
        0:w
    ]

    total_activation = (
        cam.sum() + 1e-12
    )

    center_x = (
        (xx * cam).sum()
        /
        total_activation
    )

    center_y = (
        (yy * cam).sum()
        /
        total_activation
    )

    variance = (
        (
            (
                xx - center_x
            ) ** 2
            +
            (
                yy - center_y
            ) ** 2
        )
        *
        cam
    ).sum() / total_activation

    spatial_spread = np.sqrt(
        variance
    )

    return {
        "cam_entropy": normalized_entropy,
        "top1_concentration": top1_concentration,
        "top3_concentration": top3_concentration,
        "top5_concentration": top5_concentration,
        "spatial_spread": spatial_spread
    }


# ============================================================
# 6. Analyze all KL1 validation samples
# ============================================================

records = []

print("=" * 70)
print("QUANTITATIVE GRAD-CAM ANALYSIS")
print("=" * 70)

kl1_count = 0

for idx in range(
    len(val_dataset)
):

    # Original validation label
    image_tensor, true_label = (
        val_dataset[idx]
    )

    # Only KL1
    if true_label != 1:
        continue

    kl1_count += 1

    image = (
        image_tensor
        .unsqueeze(0)
        .to(device)
    )

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    with torch.no_grad():

        logits = gradcam_model(
            image
        )

        predicted_class = int(
            torch.argmax(
                logits,
                dim=1
            ).item()
        )

    # --------------------------------------------------------
    # Generate CAM for predicted class
    # --------------------------------------------------------

    _, cam = cam_generator.generate(
        image,
        predicted_class
    )

    metrics = cam_metrics(
        cam
    )

    # --------------------------------------------------------
    # Error group
    # --------------------------------------------------------

    if predicted_class == 1:

        group = "KL1_correct"

    elif predicted_class == 0:

        group = "KL1_to_KL0"

    elif predicted_class == 2:

        group = "KL1_to_KL2"

    else:

        group = "KL1_to_other"

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    record = {
        "dataset_index": idx,
        "true_label": true_label,
        "predicted_label": predicted_class,
        "group": group
    }

    record.update(
        metrics
    )

    records.append(
        record
    )


cam_generator.close()


# ============================================================
# 7. DataFrame
# ============================================================

cam_df = pd.DataFrame(
    records
)


# ============================================================
# 8. Group summary
# ============================================================

summary = (
    cam_df
    .groupby("group")
    [
        [
            "cam_entropy",
            "top1_concentration",
            "top3_concentration",
            "top5_concentration",
            "spatial_spread"
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "std",
            "median"
        ]
    )
)


# ============================================================
# 9. Print summary
# ============================================================

print(
    f"\nTotal KL1 validation samples: "
    f"{kl1_count}"
)

print("\n" + "=" * 70)
print("CAM METRICS BY GROUP")
print("=" * 70)

print(
    summary.round(4).to_string()
)


# ============================================================
# 10. Simple compact comparison
# ============================================================

compact_summary = (
    cam_df
    .groupby("group")
    [
        [
            "cam_entropy",
            "top3_concentration",
            "spatial_spread"
        ]
    ]
    .mean()
    .round(4)
)

print("\n" + "=" * 70)
print("COMPACT GROUP COMPARISON")
print("=" * 70)

print(
    compact_summary.to_string()
)


# ============================================================
# 11. Save results
# ============================================================

CAM_RESULTS_PATH = (
    "/content/kl1_gradcam_quantitative.csv"
)

CAM_SUMMARY_PATH = (
    "/content/kl1_gradcam_summary.csv"
)

cam_df.to_csv(
    CAM_RESULTS_PATH,
    index=False
)

summary.to_csv(
    CAM_SUMMARY_PATH
)


# ============================================================
# 12. Distribution counts
# ============================================================

print("\n" + "=" * 70)
print("KL1 PREDICTION DISTRIBUTION")
print("=" * 70)

print(
    cam_df["group"]
    .value_counts()
    .sort_index()
)

print("\nSaved:")
print(CAM_RESULTS_PATH)
print(CAM_SUMMARY_PATH)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 19 — t-SNE ANALYSIS OF DEEP FEATURES
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torchvision import models
from sklearn.manifold import TSNE


# ============================================================
# 1. Feature extractor
# ============================================================

feature_model = models.resnet18(
    weights=None
)

feature_model.fc = nn.Linear(
    feature_model.fc.in_features,
    5
)

checkpoint_tsne = torch.load(
    BASELINE_CHECKPOINT,
    map_location=device,
    weights_only=False
)

feature_model.load_state_dict(
    checkpoint_tsne["model_state_dict"]
)

feature_model = feature_model.to(device)
feature_model.eval()


# ============================================================
# 2. Remove final classification layer
# ============================================================

feature_extractor = nn.Sequential(
    *list(feature_model.children())[:-1]
)

feature_extractor = feature_extractor.to(device)
feature_extractor.eval()


# ============================================================
# 3. Extract validation features
# ============================================================

all_features = []
all_labels = []

print("=" * 70)
print("EXTRACTING VALIDATION FEATURES")
print("=" * 70)

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        features = feature_extractor(
            images
        )

        # [B, 512, 1, 1] -> [B, 512]
        features = features.flatten(
            start_dim=1
        )

        all_features.append(
            features.cpu().numpy()
        )

        all_labels.append(
            labels.numpy()
        )


all_features = np.concatenate(
    all_features,
    axis=0
)

all_labels = np.concatenate(
    all_labels,
    axis=0
)

print(
    "Feature matrix:",
    all_features.shape
)

print(
    "Labels:",
    all_labels.shape
)


# ============================================================
# 4. Focus on KL0 / KL1 / KL2
# ============================================================

early_mask = np.isin(
    all_labels,
    [0, 1, 2]
)

X_early = all_features[
    early_mask
]

y_early = all_labels[
    early_mask
]

print("\nEarly-class samples:")
print(
    pd.Series(y_early)
    .value_counts()
    .sort_index()
)


# ============================================================
# 5. t-SNE
# ============================================================

print("\n" + "=" * 70)
print("RUNNING t-SNE")
print("=" * 70)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=2026
)

X_tsne = tsne.fit_transform(
    X_early
)


# ============================================================
# 6. Create dataframe
# ============================================================

tsne_df = pd.DataFrame({
    "x": X_tsne[:, 0],
    "y": X_tsne[:, 1],
    "label": y_early
})


# ============================================================
# 7. Plot
# ============================================================

plt.figure(
    figsize=(10, 8)
)

for class_id in [0, 1, 2]:

    subset = tsne_df[
        tsne_df["label"] == class_id
    ]

    plt.scatter(
        subset["x"],
        subset["y"],
        label=f"KL{class_id}",
        alpha=0.65,
        s=25
    )

plt.title(
    "t-SNE of ResNet18 Deep Features — KL0 / KL1 / KL2"
)

plt.xlabel(
    "t-SNE dimension 1"
)

plt.ylabel(
    "t-SNE dimension 2"
)

plt.legend()

plt.grid(
    alpha=0.2
)

plt.tight_layout()

plt.show()


# ============================================================
# 8. Save
# ============================================================

TSNE_PATH = (
    "/content/tsne_early_features_seed2026.csv"
)

tsne_df.to_csv(
    TSNE_PATH,
    index=False
)

print(
    "\nSaved:",
    TSNE_PATH
)

In [ ]:
# ============================================================
# CELL 20 — Install and verify CORAL
# ============================================================

!pip install -q coral-pytorch


# ============================================================
# Imports
# ============================================================

import torch

from coral_pytorch.layers import CoralLayer
from coral_pytorch.losses import CoralLoss
from coral_pytorch.dataset import (
    levels_from_labelbatch,
    proba_to_label
)


# ============================================================
# Check version
# ============================================================

import coral_pytorch

print("=" * 70)
print("CORAL-PYTORCH CHECK")
print("=" * 70)

print("Package version:", coral_pytorch.__version__)


# ============================================================
# Test CORAL layer
# ============================================================

NUM_CLASSES = 5
FEATURES = 512
BATCH_SIZE = 4

coral_layer = CoralLayer(
    size_in=FEATURES,
    num_classes=NUM_CLASSES
).to(device)

dummy_features = torch.randn(
    BATCH_SIZE,
    FEATURES,
    device=device
)

dummy_logits = coral_layer(
    dummy_features
)

print("\nFeature shape:")
print(dummy_features.shape)

print("\nCORAL logits shape:")
print(dummy_logits.shape)

assert dummy_logits.shape == (
    BATCH_SIZE,
    NUM_CLASSES - 1
)


# ============================================================
# Test label encoding
# ============================================================

dummy_labels = torch.tensor(
    [0, 1, 2, 3],
    dtype=torch.long
)

dummy_levels = levels_from_labelbatch(
    dummy_labels,
    num_classes=NUM_CLASSES
)

print("\nLabels:")
print(dummy_labels)

print("\nCORAL levels:")
print(dummy_levels)

assert dummy_levels.shape == (
    BATCH_SIZE,
    NUM_CLASSES - 1
)


# ============================================================
# Test loss
# ============================================================

criterion_coral = CoralLoss()

loss = criterion_coral(
    dummy_logits,
    dummy_levels.to(device)
)

print("\nCORAL loss:")
print(loss.item())

assert torch.isfinite(loss)


# ============================================================
# Test decoding
# ============================================================

dummy_probabilities = torch.sigmoid(
    dummy_logits
)

dummy_predictions = proba_to_label(
    dummy_probabilities
)

print("\nPredicted probabilities:")
print(dummy_probabilities)

print("\nDecoded predictions:")
print(dummy_predictions)

print("\n" + "=" * 70)
print("CORAL SANITY CHECK PASSED")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 21 — CORAL MODEL SETUP
# Same backbone / preprocessing / sampler / seed
# Only output head + loss changed
# ============================================================

import torch
import torch.nn as nn

from torchvision import models
from coral_pytorch.layers import CoralLayer
from coral_pytorch.losses import CoralLoss


# ============================================================
# 1. Create ResNet18 backbone
# ============================================================

coral_model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Number of features coming from ResNet18
num_features = coral_model.fc.in_features

print("ResNet18 feature dimension:", num_features)


# ============================================================
# 2. Replace classification head with CORAL layer
# ============================================================

coral_model.fc = CoralLayer(
    size_in=num_features,
    num_classes=5
)

coral_model = coral_model.to(device)


# ============================================================
# 3. CORAL loss
# ============================================================

coral_criterion = CoralLoss()


# ============================================================
# 4. Optimizer
# ============================================================

coral_optimizer = torch.optim.Adam(
    coral_model.parameters(),
    lr=1e-4,
    weight_decay=0
)


# ============================================================
# 5. Basic verification
# ============================================================

dummy_input = torch.randn(
    2,
    3,
    224,
    224,
    device=device
)

with torch.no_grad():

    dummy_output = coral_model(
        dummy_input
    )

print("\nModel output shape:")
print(dummy_output.shape)

assert dummy_output.shape == (
    2,
    4
)

print("\n" + "=" * 70)
print("CORAL MODEL READY")
print("=" * 70)

print("Backbone: ResNet18 pretrained")
print("Output: 4 ordinal logits")
print("Classes: 5")
print("Loss: CoralLoss")
print("Optimizer: Adam")
print("Learning rate: 1e-4")
print("Weight decay: 0")
print("Device:", device)

In [ ]:
# ============================================================
# CELL 22 — TRAIN CORAL
# ============================================================

import numpy as np
import torch
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    cohen_kappa_score
)

from coral_pytorch.dataset import (
    levels_from_labelbatch,
    proba_to_label
)


# ============================================================
# 1. Training configuration
# ============================================================

NUM_EPOCHS = 10

best_macro_f1 = -1.0
best_epoch = -1

best_state_dict = None
history = []


# ============================================================
# 2. Validation function
# ============================================================

def evaluate_coral_model(
    model,
    loader
):

    model.eval()

    all_true = []
    all_pred = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            # CORAL target encoding
            levels = levels_from_labelbatch(
                labels,
                num_classes=5
            ).to(device)

            # Forward
            logits = model(images)

            # Loss
            loss = coral_criterion(
                logits,
                levels
            )

            batch_size = (
                images.size(0)
            )

            total_loss += (
                loss.item() *
                batch_size
            )

            total_samples += batch_size

            # Decode ordinal predictions
            probabilities = torch.sigmoid(
                logits
            )

            predictions = proba_to_label(
                probabilities
            )

            all_true.extend(
                labels.cpu().numpy()
            )

            all_pred.extend(
                predictions.cpu().numpy()
            )

    avg_loss = (
        total_loss /
        total_samples
    )

    all_true = np.array(
        all_true
    )

    all_pred = np.array(
        all_pred
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        all_true,
        all_pred
    )

    balanced_acc = balanced_accuracy_score(
        all_true,
        all_pred
    )

    macro_f1 = f1_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0
    )

    mae = mean_absolute_error(
        all_true,
        all_pred
    )

    qwk = cohen_kappa_score(
        all_true,
        all_pred,
        weights="quadratic"
    )

    # Adjacent accuracy
    adjacent_acc = np.mean(
        np.abs(
            all_true - all_pred
        ) <= 1
    )

    per_class_f1 = f1_score(
        all_true,
        all_pred,
        average=None,
        labels=[0, 1, 2, 3, 4],
        zero_division=0
    )

    per_class_recall = []

    for class_id in range(5):

        true_mask = (
            all_true == class_id
        )

        if true_mask.sum() == 0:

            recall = 0.0

        else:

            recall = np.mean(
                all_pred[true_mask]
                == class_id
            )

        per_class_recall.append(
            recall
        )

    cm = confusion_matrix(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3, 4]
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "balanced_acc": balanced_acc,
        "macro_f1": macro_f1,
        "mae": mae,
        "qwk": qwk,
        "adjacent_acc": adjacent_acc,
        "per_class_f1": per_class_f1,
        "per_class_recall": np.array(
            per_class_recall
        ),
        "confusion_matrix": cm,
        "true": all_true,
        "pred": all_pred
    }


# ============================================================
# 3. Training loop
# ============================================================

print("=" * 70)
print("STARTING CORAL TRAINING")
print("=" * 70)

for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    coral_model.train()

    running_loss = 0.0
    sample_count = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # CORAL targets
        levels = levels_from_labelbatch(
            labels,
            num_classes=5
        ).to(device)

        # Clear gradients
        coral_optimizer.zero_grad(
            set_to_none=True
        )

        # Forward
        logits = coral_model(
            images
        )

        # CORAL loss
        loss = coral_criterion(
            logits,
            levels
        )

        # Backpropagation
        loss.backward()

        # Update weights
        coral_optimizer.step()

        batch_size = (
            images.size(0)
        )

        running_loss += (
            loss.item() *
            batch_size
        )

        sample_count += batch_size

    train_loss = (
        running_loss /
        sample_count
    )


    # ========================================================
    # Validation
    # ========================================================

    val_results = evaluate_coral_model(
        coral_model,
        val_loader
    )


    # ========================================================
    # Save history
    # ========================================================

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_results["loss"],
        "val_accuracy": val_results["accuracy"],
        "val_balanced_acc": val_results["balanced_acc"],
        "val_macro_f1": val_results["macro_f1"],
        "val_mae": val_results["mae"],
        "val_qwk": val_results["qwk"],
        "val_adjacent_acc": val_results["adjacent_acc"]
    }

    history.append(
        epoch_record
    )


    # ========================================================
    # Print results
    # ========================================================

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_results['loss']:.4f} | "
        f"Acc: {val_results['accuracy']:.4f} | "
        f"Bal Acc: {val_results['balanced_acc']:.4f} | "
        f"Macro F1: {val_results['macro_f1']:.4f} | "
        f"MAE: {val_results['mae']:.4f} | "
        f"QWK: {val_results['qwk']:.4f} | "
        f"Adjacent: {val_results['adjacent_acc']:.4f}"
    )


    # ========================================================
    # Best model = Macro F1
    # ========================================================

    if (
        val_results["macro_f1"]
        >
        best_macro_f1
    ):

        best_macro_f1 = (
            val_results["macro_f1"]
        )

        best_epoch = epoch

        best_state_dict = {
            key: value.detach().cpu().clone()
            for key, value
            in coral_model.state_dict().items()
        }

        print(
            f"  --> New BEST Macro F1: "
            f"{best_macro_f1:.4f}"
        )


# ============================================================
# 4. Restore best model
# ============================================================

coral_model.load_state_dict(
    best_state_dict
)

coral_model = coral_model.to(device)


# ============================================================
# 5. Final evaluation of best validation model
# ============================================================

best_results = evaluate_coral_model(
    coral_model,
    val_loader
)


# ============================================================
# 6. Print final report
# ============================================================

print("\n" + "=" * 70)
print("BEST CORAL VALIDATION RESULT")
print("=" * 70)

print(
    f"Best epoch       : {best_epoch}"
)

print(
    f"Accuracy         : {best_results['accuracy']:.4f}"
)

print(
    f"Balanced Accuracy: {best_results['balanced_acc']:.4f}"
)

print(
    f"Macro F1         : {best_results['macro_f1']:.4f}"
)

print(
    f"MAE              : {best_results['mae']:.4f}"
)

print(
    f"QWK              : {best_results['qwk']:.4f}"
)

print(
    f"Adjacent Accuracy: {best_results['adjacent_acc']:.4f}"
)


# ============================================================
# 7. Per-class metrics
# ============================================================

print("\n" + "=" * 70)
print("PER-CLASS RESULTS")
print("=" * 70)

for class_id in range(5):

    print(
        f"KL{class_id}: "
        f"F1={best_results['per_class_f1'][class_id]:.4f} | "
        f"Recall={best_results['per_class_recall'][class_id]:.4f}"
    )


# ============================================================
# 8. Confusion matrix
# ============================================================

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(
    best_results["confusion_matrix"]
)


# ============================================================
# 9. Save history
# ============================================================

history_df = pd.DataFrame(
    history
)

history_path = (
    "/content/coral_training_history_seed2026.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

print("\nSaved history:")
print(history_path)


# ============================================================
# 10. Save best model
# ============================================================

coral_checkpoint_path = (
    "/content/best_resnet18_coral_seed2026.pth"
)

torch.save(
    {
        "model_state_dict":
            coral_model.state_dict(),

        "best_epoch":
            best_epoch,

        "macro_f1":
            best_results["macro_f1"],

        "balanced_acc":
            best_results["balanced_acc"],

        "accuracy":
            best_results["accuracy"],

        "mae":
            best_results["mae"],

        "qwk":
            best_results["qwk"],

        "adjacent_acc":
            best_results["adjacent_acc"],

        "seed":
            2026
    },
    coral_checkpoint_path
)

print(
    "\nBest checkpoint saved:"
)

print(
    coral_checkpoint_path
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 23 — 448x448 DATA LOADERS
# Only resolution changes from the current benchmark
# ============================================================

import os
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import cv2


# ============================================================
# 1. Seed
# ============================================================

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. CLAHE
# ============================================================

class CLAHETransform:

    def __init__(
        self,
        clip_limit=2.0,
        grid_size=(8, 8)
    ):
        self.clip_limit = clip_limit
        self.grid_size = grid_size

    def __call__(self, img):

        img = np.array(img)

        # RGB -> grayscale
        if len(img.shape) == 3:
            img = cv2.cvtColor(
                img,
                cv2.COLOR_RGB2GRAY
            )

        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.grid_size
        )

        img = clahe.apply(img)

        # grayscale -> RGB
        img = cv2.cvtColor(
            img,
            cv2.COLOR_GRAY2RGB
        )

        return Image.fromarray(img)


# ============================================================
# 3. 448x448 transforms
# ============================================================

train_transform_448 = transforms.Compose([

    transforms.Resize(
        (448, 448)
    ),

    CLAHETransform(
        clip_limit=2.0,
        grid_size=(8, 8)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=5
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


val_transform_448 = transforms.Compose([

    transforms.Resize(
        (448, 448)
    ),

    CLAHETransform(
        clip_limit=2.0,
        grid_size=(8, 8)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 4. Recreate datasets
# ============================================================

train_dataset_448 = KneeOsteoarthritisDataset(
    root_dir=os.path.join(
        DATASET_PATH,
        "train"
    ),
    transform=train_transform_448
)

val_dataset_448 = KneeOsteoarthritisDataset(
    root_dir=os.path.join(
        DATASET_PATH,
        "val"
    ),
    transform=val_transform_448
)


# ============================================================
# 5. WeightedRandomSampler
# ============================================================

labels_448 = np.array(
    train_dataset_448.labels
)

class_counts_448 = np.bincount(
    labels_448,
    minlength=5
)

class_weights_448 = (
    1.0 /
    class_counts_448
)

sample_weights_448 = (
    class_weights_448[
        labels_448
    ]
)

sampler_generator_448 = torch.Generator()

sampler_generator_448.manual_seed(
    SEED
)

sampler_448 = WeightedRandomSampler(
    weights=torch.DoubleTensor(
        sample_weights_448
    ),
    num_samples=len(
        sample_weights_448
    ),
    replacement=True,
    generator=sampler_generator_448
)


# ============================================================
# 6. DataLoader generator
# ============================================================

loader_generator_448 = torch.Generator()

loader_generator_448.manual_seed(
    SEED
)


def seed_worker_448(worker_id):

    worker_seed = (
        torch.initial_seed()
        % 2**32
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


# ============================================================
# 7. DataLoaders
# ============================================================

train_loader_448 = DataLoader(
    train_dataset_448,
    batch_size=16,
    sampler=sampler_448,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker_448,
    generator=loader_generator_448
)

val_loader_448 = DataLoader(
    val_dataset_448,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker_448,
    generator=loader_generator_448
)


# ============================================================
# 8. Verification
# ============================================================

images_448, labels_448_batch = next(
    iter(train_loader_448)
)

print("=" * 70)
print("448x448 DATA LOADER CHECK")
print("=" * 70)

print(
    "Train samples:",
    len(train_dataset_448)
)

print(
    "Validation samples:",
    len(val_dataset_448)
)

print(
    "Batch shape:",
    images_448.shape
)

print(
    "Labels shape:",
    labels_448_batch.shape
)

print(
    "Image resolution:",
    images_448.shape[-2:]
)

assert images_448.shape == (
    16,
    3,
    448,
    448
)

print("\nResolution check PASSED.")

In [ ]:
# ============================================================
# CELL 23A — RESTORE KNEE OA DATASET CLASS
# ============================================================

import os
from PIL import Image
from torch.utils.data import Dataset


class KneeOsteoarthritisDataset(Dataset):

    def __init__(
        self,
        root_dir,
        transform=None
    ):

        self.root_dir = root_dir
        self.transform = transform

        self.image_paths = []
        self.labels = []

        # Class folders: 0, 1, 2, 3, 4
        for class_name in sorted(
            os.listdir(root_dir)
        ):

            class_dir = os.path.join(
                root_dir,
                class_name
            )

            if not os.path.isdir(
                class_dir
            ):
                continue

            # Ignore anything that is not 0-4
            if class_name not in [
                "0",
                "1",
                "2",
                "3",
                "4"
            ]:
                continue

            label = int(class_name)

            for filename in sorted(
                os.listdir(class_dir)
            ):

                file_path = os.path.join(
                    class_dir,
                    filename
                )

                if filename.lower().endswith(
                    (
                        ".png",
                        ".jpg",
                        ".jpeg"
                    )
                ):

                    self.image_paths.append(
                        file_path
                    )

                    self.labels.append(
                        label
                    )

    def __len__(self):

        return len(
            self.image_paths
        )

    def __getitem__(
        self,
        idx
    ):

        image_path = (
            self.image_paths[idx]
        )

        label = self.labels[idx]

        image = Image.open(
            image_path
        ).convert("RGB")

        if self.transform is not None:

            image = self.transform(
                image
            )

        return image, label


# ============================================================
# Verify Dataset class
# ============================================================

print("=" * 70)
print("DATASET CLASS RESTORED")
print("=" * 70)

print(
    "Dataset path:",
    DATASET_PATH
)

print(
    "Train directory exists:",
    os.path.exists(
        os.path.join(
            DATASET_PATH,
            "train"
        )
    )
)

print(
    "Validation directory exists:",
    os.path.exists(
        os.path.join(
            DATASET_PATH,
            "val"
        )
    )
)

In [ ]:
# ============================================================
# CELL 24 — RESNET18 448x448 MODEL
# ============================================================

import torch
import torch.nn as nn
from torchvision import models


# ============================================================
# 1. Create pretrained ResNet18
# ============================================================

model_448 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)


# ============================================================
# 2. Replace final layer
# ============================================================

num_features_448 = model_448.fc.in_features

model_448.fc = nn.Linear(
    num_features_448,
    5
)


# ============================================================
# 3. Move to GPU
# ============================================================

model_448 = model_448.to(device)


# ============================================================
# 4. Loss
# ============================================================

criterion_448 = nn.CrossEntropyLoss()


# ============================================================
# 5. Optimizer
# ============================================================

optimizer_448 = torch.optim.Adam(
    model_448.parameters(),
    lr=1e-4,
    weight_decay=0
)


# ============================================================
# 6. Verification
# ============================================================

dummy_input_448 = torch.randn(
    2,
    3,
    448,
    448,
    device=device
)

with torch.no_grad():

    dummy_output_448 = model_448(
        dummy_input_448
    )


print("=" * 70)
print("448x448 MODEL CHECK")
print("=" * 70)

print(
    "Input shape:",
    dummy_input_448.shape
)

print(
    "Output shape:",
    dummy_output_448.shape
)

print(
    "Number of classes:",
    dummy_output_448.shape[1]
)

print(
    "Loss: CrossEntropyLoss"
)

print(
    "Optimizer: Adam"
)

print(
    "Learning rate: 1e-4"
)

print(
    "Weight decay: 0"
)

print(
    "Device:",
    device
)

assert dummy_output_448.shape == (
    2,
    5
)

print("\nModel check PASSED.")

In [ ]:
# ============================================================
# CELL 25 — TRAIN RESNET18 AT 448x448
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    cohen_kappa_score
)


# ============================================================
# 1. Training configuration
# ============================================================

NUM_EPOCHS_448 = 10

best_macro_f1_448 = -1.0
best_epoch_448 = -1

best_state_dict_448 = None

history_448 = []


# ============================================================
# 2. Validation function
# ============================================================

def evaluate_448(
    model,
    loader,
    criterion
):

    model.eval()

    all_true = []
    all_pred = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images
            )

            loss = criterion(
                outputs,
                labels
            )

            batch_size = (
                images.size(0)
            )

            total_loss += (
                loss.item() *
                batch_size
            )

            total_samples += (
                batch_size
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_true.extend(
                labels.cpu().numpy()
            )

            all_pred.extend(
                predictions.cpu().numpy()
            )

    avg_loss = (
        total_loss /
        total_samples
    )

    all_true = np.array(
        all_true
    )

    all_pred = np.array(
        all_pred
    )

    accuracy = accuracy_score(
        all_true,
        all_pred
    )

    balanced_acc = (
        balanced_accuracy_score(
            all_true,
            all_pred
        )
    )

    macro_f1 = f1_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0
    )

    mae = mean_absolute_error(
        all_true,
        all_pred
    )

    qwk = cohen_kappa_score(
        all_true,
        all_pred,
        weights="quadratic"
    )

    adjacent_acc = np.mean(
        np.abs(
            all_true - all_pred
        ) <= 1
    )

    per_class_f1 = f1_score(
        all_true,
        all_pred,
        average=None,
        labels=[0, 1, 2, 3, 4],
        zero_division=0
    )

    per_class_recall = []

    for class_id in range(5):

        mask = (
            all_true == class_id
        )

        if mask.sum() == 0:

            recall = 0.0

        else:

            recall = np.mean(
                all_pred[mask]
                == class_id
            )

        per_class_recall.append(
            recall
        )

    cm = confusion_matrix(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3, 4]
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "balanced_acc": balanced_acc,
        "macro_f1": macro_f1,
        "mae": mae,
        "qwk": qwk,
        "adjacent_acc": adjacent_acc,
        "per_class_f1": np.array(
            per_class_f1
        ),
        "per_class_recall": np.array(
            per_class_recall
        ),
        "confusion_matrix": cm
    }


# ============================================================
# 3. Training
# ============================================================

print("=" * 70)
print("STARTING 448x448 TRAINING")
print("=" * 70)

for epoch in range(
    1,
    NUM_EPOCHS_448 + 1
):

    model_448.train()

    running_loss = 0.0
    sample_count = 0

    for images, labels in train_loader_448:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_448.zero_grad(
            set_to_none=True
        )

        outputs = model_448(
            images
        )

        loss = criterion_448(
            outputs,
            labels
        )

        loss.backward()

        optimizer_448.step()

        batch_size = (
            images.size(0)
        )

        running_loss += (
            loss.item() *
            batch_size
        )

        sample_count += (
            batch_size
        )

    train_loss = (
        running_loss /
        sample_count
    )


    # ========================================================
    # Validation
    # ========================================================

    results_448 = evaluate_448(
        model_448,
        val_loader_448,
        criterion_448
    )


    # ========================================================
    # History
    # ========================================================

    history_448.append({

        "epoch": epoch,

        "train_loss":
            train_loss,

        "val_loss":
            results_448["loss"],

        "val_accuracy":
            results_448["accuracy"],

        "val_balanced_acc":
            results_448["balanced_acc"],

        "val_macro_f1":
            results_448["macro_f1"],

        "val_mae":
            results_448["mae"],

        "val_qwk":
            results_448["qwk"],

        "val_adjacent_acc":
            results_448["adjacent_acc"]

    })


    # ========================================================
    # Print
    # ========================================================

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS_448} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {results_448['loss']:.4f} | "
        f"Acc: {results_448['accuracy']:.4f} | "
        f"Bal Acc: {results_448['balanced_acc']:.4f} | "
        f"Macro F1: {results_448['macro_f1']:.4f} | "
        f"MAE: {results_448['mae']:.4f} | "
        f"QWK: {results_448['qwk']:.4f} | "
        f"Adjacent: {results_448['adjacent_acc']:.4f}"
    )


    # ========================================================
    # Best model
    # ========================================================

    if (
        results_448["macro_f1"]
        >
        best_macro_f1_448
    ):

        best_macro_f1_448 = (
            results_448["macro_f1"]
        )

        best_epoch_448 = epoch

        best_state_dict_448 = {

            key: value.detach()
            .cpu()
            .clone()

            for key, value
            in model_448.state_dict().items()

        }

        print(
            f"  --> New BEST Macro F1: "
            f"{best_macro_f1_448:.4f}"
        )


# ============================================================
# 4. Restore best model
# ============================================================

model_448.load_state_dict(
    best_state_dict_448
)

model_448 = model_448.to(device)


# ============================================================
# 5. Final validation
# ============================================================

best_results_448 = evaluate_448(
    model_448,
    val_loader_448,
    criterion_448
)


# ============================================================
# 6. Final report
# ============================================================

print("\n" + "=" * 70)
print("BEST 448x448 VALIDATION RESULT")
print("=" * 70)

print(
    f"Best epoch       : "
    f"{best_epoch_448}"
)

print(
    f"Accuracy         : "
    f"{best_results_448['accuracy']:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{best_results_448['balanced_acc']:.4f}"
)

print(
    f"Macro F1         : "
    f"{best_results_448['macro_f1']:.4f}"
)

print(
    f"MAE              : "
    f"{best_results_448['mae']:.4f}"
)

print(
    f"QWK              : "
    f"{best_results_448['qwk']:.4f}"
)

print(
    f"Adjacent Accuracy: "
    f"{best_results_448['adjacent_acc']:.4f}"
)


# ============================================================
# 7. Per-class
# ============================================================

print("\n" + "=" * 70)
print("PER-CLASS RESULTS")
print("=" * 70)

for class_id in range(5):

    print(
        f"KL{class_id}: "
        f"F1="
        f"{best_results_448['per_class_f1'][class_id]:.4f} | "
        f"Recall="
        f"{best_results_448['per_class_recall'][class_id]:.4f}"
    )


# ============================================================
# 8. Confusion matrix
# ============================================================

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(
    best_results_448[
        "confusion_matrix"
    ]
)


# ============================================================
# 9. Save history
# ============================================================

history_448_df = pd.DataFrame(
    history_448
)

history_448_path = (
    "/content/resnet18_448_history_seed2026.csv"
)

history_448_df.to_csv(
    history_448_path,
    index=False
)


# ============================================================
# 10. Save checkpoint
# ============================================================

checkpoint_448_path = (
    "/content/"
    "best_resnet18_448_seed2026.pth"
)

torch.save(
    {
        "model_state_dict":
            model_448.state_dict(),

        "best_epoch":
            best_epoch_448,

        "macro_f1":
            best_results_448[
                "macro_f1"
            ],

        "balanced_acc":
            best_results_448[
                "balanced_acc"
            ],

        "accuracy":
            best_results_448[
                "accuracy"
            ],

        "mae":
            best_results_448[
                "mae"
            ],

        "qwk":
            best_results_448[
                "qwk"
            ],

        "adjacent_acc":
            best_results_448[
                "adjacent_acc"
            ],

        "seed":
            2026,

        "resolution":
            448

    },
    checkpoint_448_path
)


print("\nSaved:")
print(history_448_path)
print(checkpoint_448_path)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 26 — 448x448 WITH SEED=42
# Reproducibility check
# ============================================================

import os
import random
import numpy as np
import torch

from torch.utils.data import DataLoader, WeightedRandomSampler


# ============================================================
# 1. Seed = 42
# ============================================================

SEED_42 = 42

random.seed(SEED_42)
np.random.seed(SEED_42)
torch.manual_seed(SEED_42)
torch.cuda.manual_seed_all(SEED_42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 2. Re-create datasets with the SAME 448 transforms
# ============================================================

train_dataset_448_s42 = KneeOsteoarthritisDataset(
    root_dir=os.path.join(
        DATASET_PATH,
        "train"
    ),
    transform=train_transform_448
)

val_dataset_448_s42 = KneeOsteoarthritisDataset(
    root_dir=os.path.join(
        DATASET_PATH,
        "val"
    ),
    transform=val_transform_448
)


# ============================================================
# 3. WeightedRandomSampler
# ============================================================

labels_s42 = np.array(
    train_dataset_448_s42.labels
)

class_counts_s42 = np.bincount(
    labels_s42,
    minlength=5
)

class_weights_s42 = (
    1.0 / class_counts_s42
)

sample_weights_s42 = (
    class_weights_s42[
        labels_s42
    ]
)

sampler_generator_s42 = torch.Generator()

sampler_generator_s42.manual_seed(
    SEED_42
)

sampler_s42 = WeightedRandomSampler(
    weights=torch.DoubleTensor(
        sample_weights_s42
    ),
    num_samples=len(
        sample_weights_s42
    ),
    replacement=True,
    generator=sampler_generator_s42
)


# ============================================================
# 4. Worker seeding
# ============================================================

loader_generator_s42 = torch.Generator()

loader_generator_s42.manual_seed(
    SEED_42
)


def seed_worker_s42(worker_id):

    worker_seed = (
        torch.initial_seed()
        % 2**32
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


# ============================================================
# 5. DataLoaders
# ============================================================

train_loader_448_s42 = DataLoader(
    train_dataset_448_s42,
    batch_size=16,
    sampler=sampler_s42,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker_s42,
    generator=loader_generator_s42
)

val_loader_448_s42 = DataLoader(
    val_dataset_448_s42,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker_s42,
    generator=loader_generator_s42
)


# ============================================================
# 6. Verification
# ============================================================

x_s42, y_s42 = next(
    iter(train_loader_448_s42)
)

print("=" * 70)
print("448x448 SEED=42 DATA LOADER CHECK")
print("=" * 70)

print(
    "Seed:",
    SEED_42
)

print(
    "Train:",
    len(train_dataset_448_s42)
)

print(
    "Validation:",
    len(val_dataset_448_s42)
)

print(
    "Batch shape:",
    x_s42.shape
)

print(
    "Resolution:",
    x_s42.shape[-2:]
)

assert x_s42.shape == (
    16,
    3,
    448,
    448
)

print("\nSeed 42 loader check PASSED.")

In [ ]:
# ============================================================
# CELL 27 — RESNET18 448x448 | SEED=42
# ============================================================

import torch
import torch.nn as nn
from torchvision import models


# ============================================================
# 1. Re-apply seed
# ============================================================

SEED_42 = 42

torch.manual_seed(SEED_42)
torch.cuda.manual_seed_all(SEED_42)


# ============================================================
# 2. Create pretrained ResNet18
# ============================================================

model_448_s42 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)


# ============================================================
# 3. Replace classification head
# ============================================================

num_features_s42 = (
    model_448_s42.fc.in_features
)

model_448_s42.fc = nn.Linear(
    num_features_s42,
    5
)


# ============================================================
# 4. Move to GPU
# ============================================================

model_448_s42 = model_448_s42.to(device)


# ============================================================
# 5. Loss
# ============================================================

criterion_448_s42 = (
    nn.CrossEntropyLoss()
)


# ============================================================
# 6. Optimizer
# ============================================================

optimizer_448_s42 = torch.optim.Adam(
    model_448_s42.parameters(),
    lr=1e-4,
    weight_decay=0
)


# ============================================================
# 7. Verification
# ============================================================

dummy_input_s42 = torch.randn(
    2,
    3,
    448,
    448,
    device=device
)

with torch.no_grad():

    dummy_output_s42 = (
        model_448_s42(
            dummy_input_s42
        )
    )


print("=" * 70)
print("448x448 SEED=42 MODEL CHECK")
print("=" * 70)

print(
    "Seed:",
    SEED_42
)

print(
    "Input:",
    dummy_input_s42.shape
)

print(
    "Output:",
    dummy_output_s42.shape
)

print(
    "Loss: CrossEntropyLoss"
)

print(
    "Optimizer: Adam"
)

print(
    "Learning rate: 1e-4"
)

print(
    "Device:",
    device
)

assert dummy_output_s42.shape == (
    2,
    5
)

print("\nModel check PASSED.")

In [ ]:
# ============================================================
# CELL 28 — TRAIN RESNET18 448x448 | SEED=42
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    cohen_kappa_score
)


# ============================================================
# 1. Configuration
# ============================================================

NUM_EPOCHS_S42 = 10

best_macro_f1_s42 = -1.0
best_epoch_s42 = -1
best_state_dict_s42 = None

history_s42 = []


# ============================================================
# 2. Evaluation function
# ============================================================

def evaluate_448_s42(
    model,
    loader,
    criterion
):

    model.eval()

    all_true = []
    all_pred = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            batch_size = images.size(0)

            total_loss += (
                loss.item() * batch_size
            )

            total_samples += batch_size

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_true.extend(
                labels.cpu().numpy()
            )

            all_pred.extend(
                predictions.cpu().numpy()
            )

    avg_loss = (
        total_loss /
        total_samples
    )

    all_true = np.asarray(all_true)
    all_pred = np.asarray(all_pred)

    accuracy = accuracy_score(
        all_true,
        all_pred
    )

    balanced_acc = balanced_accuracy_score(
        all_true,
        all_pred
    )

    macro_f1 = f1_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0
    )

    mae = mean_absolute_error(
        all_true,
        all_pred
    )

    qwk = cohen_kappa_score(
        all_true,
        all_pred,
        weights="quadratic"
    )

    adjacent_acc = np.mean(
        np.abs(
            all_true - all_pred
        ) <= 1
    )

    per_class_f1 = f1_score(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3, 4],
        average=None,
        zero_division=0
    )

    per_class_recall = []

    for class_id in range(5):

        mask = (
            all_true == class_id
        )

        if mask.sum() == 0:

            recall = 0.0

        else:

            recall = np.mean(
                all_pred[mask] == class_id
            )

        per_class_recall.append(
            recall
        )

    cm = confusion_matrix(
        all_true,
        all_pred,
        labels=[0, 1, 2, 3, 4]
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "balanced_acc": balanced_acc,
        "macro_f1": macro_f1,
        "mae": mae,
        "qwk": qwk,
        "adjacent_acc": adjacent_acc,
        "per_class_f1": np.asarray(
            per_class_f1
        ),
        "per_class_recall": np.asarray(
            per_class_recall
        ),
        "confusion_matrix": cm
    }


# ============================================================
# 3. Training loop
# ============================================================

print("=" * 70)
print("STARTING 448x448 TRAINING | SEED=42")
print("=" * 70)

for epoch in range(
    1,
    NUM_EPOCHS_S42 + 1
):

    model_448_s42.train()

    running_loss = 0.0
    sample_count = 0

    for images, labels in train_loader_448_s42:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer_448_s42.zero_grad(
            set_to_none=True
        )

        outputs = model_448_s42(
            images
        )

        loss = criterion_448_s42(
            outputs,
            labels
        )

        loss.backward()

        optimizer_448_s42.step()

        batch_size = images.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        sample_count += batch_size

    train_loss = (
        running_loss /
        sample_count
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    results_s42 = evaluate_448_s42(
        model_448_s42,
        val_loader_448_s42,
        criterion_448_s42
    )

    history_s42.append({

        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": results_s42["loss"],
        "val_accuracy": results_s42["accuracy"],
        "val_balanced_acc": results_s42["balanced_acc"],
        "val_macro_f1": results_s42["macro_f1"],
        "val_mae": results_s42["mae"],
        "val_qwk": results_s42["qwk"],
        "val_adjacent_acc": results_s42["adjacent_acc"]

    })

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS_S42} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {results_s42['loss']:.4f} | "
        f"Acc: {results_s42['accuracy']:.4f} | "
        f"Bal Acc: {results_s42['balanced_acc']:.4f} | "
        f"Macro F1: {results_s42['macro_f1']:.4f} | "
        f"MAE: {results_s42['mae']:.4f} | "
        f"QWK: {results_s42['qwk']:.4f} | "
        f"Adjacent: {results_s42['adjacent_acc']:.4f}"
    )

    # --------------------------------------------------------
    # Best model by Macro F1
    # --------------------------------------------------------

    if (
        results_s42["macro_f1"]
        >
        best_macro_f1_s42
    ):

        best_macro_f1_s42 = (
            results_s42["macro_f1"]
        )

        best_epoch_s42 = epoch

        best_state_dict_s42 = {
            key: value.detach()
            .cpu()
            .clone()
            for key, value
            in model_448_s42.state_dict().items()
        }

        print(
            f"  --> New BEST Macro F1: "
            f"{best_macro_f1_s42:.4f}"
        )


# ============================================================
# 4. Restore best model
# ============================================================

model_448_s42.load_state_dict(
    best_state_dict_s42
)

model_448_s42 = model_448_s42.to(device)


# ============================================================
# 5. Final evaluation
# ============================================================

best_results_s42 = evaluate_448_s42(
    model_448_s42,
    val_loader_448_s42,
    criterion_448_s42
)


# ============================================================
# 6. Final report
# ============================================================

print("\n" + "=" * 70)
print("BEST 448x448 SEED=42 VALIDATION RESULT")
print("=" * 70)

print(
    f"Best epoch       : {best_epoch_s42}"
)

print(
    f"Accuracy         : "
    f"{best_results_s42['accuracy']:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{best_results_s42['balanced_acc']:.4f}"
)

print(
    f"Macro F1         : "
    f"{best_results_s42['macro_f1']:.4f}"
)

print(
    f"MAE              : "
    f"{best_results_s42['mae']:.4f}"
)

print(
    f"QWK              : "
    f"{best_results_s42['qwk']:.4f}"
)

print(
    f"Adjacent Accuracy: "
    f"{best_results_s42['adjacent_acc']:.4f}"
)


# ============================================================
# 7. Per-class results
# ============================================================

print("\n" + "=" * 70)
print("PER-CLASS RESULTS")
print("=" * 70)

for class_id in range(5):

    print(
        f"KL{class_id}: "
        f"F1={best_results_s42['per_class_f1'][class_id]:.4f} | "
        f"Recall={best_results_s42['per_class_recall'][class_id]:.4f}"
    )


# ============================================================
# 8. Confusion matrix
# ============================================================

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(
    best_results_s42[
        "confusion_matrix"
    ]
)


# ============================================================
# 9. Save history + checkpoint
# ============================================================

history_s42_df = pd.DataFrame(
    history_s42
)

history_s42_path = (
    "/content/"
    "resnet18_448_history_seed42.csv"
)

history_s42_df.to_csv(
    history_s42_path,
    index=False
)

checkpoint_s42_path = (
    "/content/"
    "best_resnet18_448_seed42.pth"
)

torch.save(
    {
        "model_state_dict":
            model_448_s42.state_dict(),

        "best_epoch":
            best_epoch_s42,

        "macro_f1":
            best_results_s42["macro_f1"],

        "balanced_acc":
            best_results_s42["balanced_acc"],

        "accuracy":
            best_results_s42["accuracy"],

        "mae":
            best_results_s42["mae"],

        "qwk":
            best_results_s42["qwk"],

        "adjacent_acc":
            best_results_s42["adjacent_acc"],

        "seed":
            42,

        "resolution":
            448

    },
    checkpoint_s42_path
)

print("\nSaved:")
print(history_s42_path)
print(checkpoint_s42_path)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 29 — DIRECT COMPARISON: 224 vs 448
# Focus: KL1 and early-stage classes
# ============================================================

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score


# ============================================================
# 1. Load 224 baseline checkpoint
# ============================================================

BASELINE_224_CHECKPOINT = (
    "/content/best_resnet18_sampler_seed2026.pth"
)

if not os.path.exists(
    BASELINE_224_CHECKPOINT
):
    raise FileNotFoundError(
        f"224 checkpoint not found:\n"
        f"{BASELINE_224_CHECKPOINT}"
    )


model_224_compare = models.resnet18(
    weights=None
)

model_224_compare.fc = nn.Linear(
    model_224_compare.fc.in_features,
    5
)

ckpt_224 = torch.load(
    BASELINE_224_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model_224_compare.load_state_dict(
    ckpt_224["model_state_dict"]
)

model_224_compare = (
    model_224_compare.to(device)
)

model_224_compare.eval()


# ============================================================
# 2. Helper: predictions
# ============================================================

def get_predictions(
    model,
    loader
):

    model.eval()

    all_true = []
    all_pred = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images
            )

            preds = torch.argmax(
                outputs,
                dim=1
            )

            all_true.extend(
                labels.numpy()
            )

            all_pred.extend(
                preds.cpu().numpy()
            )

    return (
        np.asarray(all_true),
        np.asarray(all_pred)
    )


# ============================================================
# 3. Get predictions from 224 baseline
# ============================================================

true_224, pred_224 = get_predictions(
    model_224_compare,
    val_loader
)


# ============================================================
# 4. Get predictions from 448 Seed 2026
# ============================================================

true_448, pred_448 = get_predictions(
    model_448,
    val_loader_448
)


# ============================================================
# 5. Verify labels are identical
# ============================================================

assert np.array_equal(
    true_224,
    true_448
)

y_true = true_224


# ============================================================
# 6. Overall comparison
# ============================================================

f1_224 = f1_score(
    y_true,
    pred_224,
    average="macro",
    zero_division=0
)

f1_448 = f1_score(
    y_true,
    pred_448,
    average="macro",
    zero_division=0
)

print("=" * 70)
print("224 vs 448")
print("=" * 70)

print(
    f"224 Macro F1 : {f1_224:.4f}"
)

print(
    f"448 Macro F1 : {f1_448:.4f}"
)


# ============================================================
# 7. Per-sample transition categories
# ============================================================

transition = []

for true_label, p224, p448 in zip(
    y_true,
    pred_224,
    pred_448
):

    correct_224 = (
        p224 == true_label
    )

    correct_448 = (
        p448 == true_label
    )

    if (
        not correct_224
        and correct_448
    ):
        category = "224_wrong_448_correct"

    elif (
        correct_224
        and not correct_448
    ):
        category = "224_correct_448_wrong"

    elif (
        correct_224
        and correct_448
    ):
        category = "both_correct"

    else:
        category = "both_wrong"

    transition.append(
        category
    )


# ============================================================
# 8. Focus on KL1
# ============================================================

kl1_mask = (
    y_true == 1
)

print("\n" + "=" * 70)
print("KL1 TRANSITIONS")
print("=" * 70)

kl1_transition_counts = (
    pd.Series(
        np.asarray(transition)[kl1_mask]
    )
    .value_counts()
)

print(
    kl1_transition_counts.to_string()
)


# ============================================================
# 9. Detailed KL1 transition table
# ============================================================

kl1_table = pd.DataFrame({
    "true": y_true[kl1_mask],
    "pred_224": pred_224[kl1_mask],
    "pred_448": pred_448[kl1_mask],
    "transition": np.asarray(
        transition
    )[kl1_mask]
})

print("\n" + "=" * 70)
print("KL1 PREDICTION CHANGES")
print("=" * 70)

change_table = (
    kl1_table[
        kl1_table["pred_224"]
        !=
        kl1_table["pred_448"]
    ]
    .groupby(
        [
            "pred_224",
            "pred_448"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

print(
    change_table.to_string(
        index=False
    )
)


# ============================================================
# 10. Important: improvements and regressions
# ============================================================

improved_kl1 = kl1_table[
    kl1_table["transition"]
    ==
    "224_wrong_448_correct"
]

regressed_kl1 = kl1_table[
    kl1_table["transition"]
    ==
    "224_correct_448_wrong"
]

print("\n" + "=" * 70)
print("KL1 IMPROVEMENTS / REGRESSIONS")
print("=" * 70)

print(
    "224 wrong → 448 correct:",
    len(improved_kl1)
)

print(
    "224 correct → 448 wrong:",
    len(regressed_kl1)
)


# ============================================================
# 11. All early-class transitions
# ============================================================

early_mask = np.isin(
    y_true,
    [0, 1, 2]
)

early_transition_counts = (
    pd.Series(
        np.asarray(transition)[early_mask]
    )
    .value_counts()
)

print("\n" + "=" * 70)
print("KL0 / KL1 / KL2 TRANSITIONS")
print("=" * 70)

print(
    early_transition_counts.to_string()
)


# ============================================================
# 12. Save comparison
# ============================================================

comparison_df = pd.DataFrame({

    "true_label":
        y_true,

    "pred_224":
        pred_224,

    "pred_448":
        pred_448,

    "transition":
        transition
})

comparison_path = (
    "/content/"
    "comparison_224_vs_448_seed2026.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print("\nSaved:")
print(comparison_path)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 30 — BUILD COMPARISON + VISUALIZE 224 vs 448
# KL1: 224 Wrong -> 448 Correct
# KL1: 224 Correct -> 448 Wrong
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import models


# ============================================================
# 1. CHECK REQUIRED OBJECTS
# ============================================================

required_objects = [
    "val_dataset",
    "val_dataset_448",
    "model_224_compare",
    "model_448"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "The following required objects are missing:\n"
        + "\n".join(
            f"- {name}"
            for name in missing_objects
        )
        + "\n\nPlease make sure the previous "
          "224/448 comparison cells have been run."
    )


# ============================================================
# 2. HELPER — INVERSE NORMALIZATION
# ============================================================

MEAN = np.array(
    [0.485, 0.456, 0.406]
)

STD = np.array(
    [0.229, 0.224, 0.225]
)


def denormalize_image(tensor):

    img = (
        tensor.detach()
        .cpu()
        .numpy()
    )

    img = np.transpose(
        img,
        (1, 2, 0)
    )

    img = (
        img * STD
        + MEAN
    )

    img = np.clip(
        img,
        0,
        1
    )

    return img


# ============================================================
# 3. GRAD-CAM CLASS
# ============================================================

class SimpleGradCAM:

    def __init__(
        self,
        model,
        target_layer
    ):

        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_handle = (
            target_layer.register_forward_hook(
                self._forward_hook
            )
        )

        self.backward_handle = (
            target_layer.register_full_backward_hook(
                self._backward_hook
            )
        )

    def _forward_hook(
        self,
        module,
        inputs,
        output
    ):

        self.activations = output

    def _backward_hook(
        self,
        module,
        grad_input,
        grad_output
    ):

        self.gradients = grad_output[0]

    def generate(
        self,
        image,
        class_id
    ):

        self.model.zero_grad(
            set_to_none=True
        )

        output = self.model(
            image
        )

        score = output[
            0,
            class_id
        ]

        score.backward()

        activations = (
            self.activations[0]
        )

        gradients = (
            self.gradients[0]
        )

        weights = gradients.mean(
            dim=(1, 2)
        )

        cam = (
            weights[:, None, None]
            * activations
        ).sum(
            dim=0
        )

        cam = torch.relu(
            cam
        )

        cam = (
            cam.detach()
            .cpu()
            .numpy()
        )

        cam_min = cam.min()
        cam_max = cam.max()

        if cam_max > cam_min:

            cam = (
                cam - cam_min
            ) / (
                cam_max - cam_min
            )

        else:

            cam = np.zeros_like(
                cam
            )

        return cam

    def close(self):

        self.forward_handle.remove()
        self.backward_handle.remove()


# ============================================================
# 4. PREDICTION FUNCTION
# ============================================================

def get_predictions(
    model,
    loader
):

    model.eval()

    all_true = []
    all_pred = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_true.extend(
                labels.numpy()
            )

            all_pred.extend(
                predictions.cpu().numpy()
            )

    return (
        np.asarray(all_true),
        np.asarray(all_pred)
    )


# ============================================================
# 5. LOAD PREDICTIONS
# ============================================================

print("=" * 70)
print("BUILDING 224 vs 448 COMPARISON")
print("=" * 70)

true_224, pred_224 = get_predictions(
    model_224_compare,
    val_loader
)

true_448, pred_448 = get_predictions(
    model_448,
    val_loader_448
)


# ============================================================
# 6. VERIFY SAME VALIDATION ORDER
# ============================================================

if not np.array_equal(
    true_224,
    true_448
):

    raise RuntimeError(
        "224 and 448 validation labels are not aligned."
    )

y_true = true_224


# ============================================================
# 7. CREATE TRANSITION LABELS
# ============================================================

transition = []

for true_label, p224, p448 in zip(
    y_true,
    pred_224,
    pred_448
):

    correct_224 = (
        p224 == true_label
    )

    correct_448 = (
        p448 == true_label
    )

    if (
        not correct_224
        and correct_448
    ):

        category = (
            "224_wrong_448_correct"
        )

    elif (
        correct_224
        and not correct_448
    ):

        category = (
            "224_correct_448_wrong"
        )

    elif (
        correct_224
        and correct_448
    ):

        category = "both_correct"

    else:

        category = "both_wrong"

    transition.append(
        category
    )


# ============================================================
# 8. BUILD COMPARISON DATAFRAME
# IMPORTANT:
# dataset_index = actual ImageFolder validation index
# ============================================================

comparison_df = pd.DataFrame({

    "dataset_index":
        np.arange(
            len(y_true)
        ),

    "true_label":
        y_true,

    "pred_224":
        pred_224,

    "pred_448":
        pred_448,

    "transition":
        transition

})


# ============================================================
# 9. SAVE COMPARISON
# ============================================================

comparison_path = (
    "/content/"
    "comparison_224_vs_448_seed2026.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)


# ============================================================
# 10. SELECT KL1 CASES
# ============================================================

improved_df = comparison_df[
    (
        comparison_df["true_label"]
        == 1
    )
    &
    (
        comparison_df["transition"]
        ==
        "224_wrong_448_correct"
    )
].copy()

regressed_df = comparison_df[
    (
        comparison_df["true_label"]
        == 1
    )
    &
    (
        comparison_df["transition"]
        ==
        "224_correct_448_wrong"
    )
].copy()


print(
    "\nKL1: 224 wrong → 448 correct:",
    len(improved_df)
)

print(
    "KL1: 224 correct → 448 wrong:",
    len(regressed_df)
)


# ============================================================
# 11. CREATE GRAD-CAM GENERATORS
# ============================================================

cam_224 = SimpleGradCAM(
    model_224_compare,
    model_224_compare.layer4[-1]
)

cam_448 = SimpleGradCAM(
    model_448,
    model_448.layer4[-1]
)


# ============================================================
# 12. PLOT CAM HELPER
# ============================================================

def plot_cam(
    ax,
    image_tensor,
    cam,
    title
):

    image = denormalize_image(
        image_tensor
    )

    ax.imshow(
        image
    )

    ax.imshow(
        cam,
        cmap="jet",
        alpha=0.45,
        extent=(
            0,
            image.shape[1],
            image.shape[0],
            0
        )
    )

    ax.set_title(
        title,
        fontsize=9
    )

    ax.axis("off")


# ============================================================
# 13. CREATE TRANSITION GRID
# ============================================================

def create_transition_grid(
    df,
    filename,
    title
):

    rows = len(df)

    if rows == 0:

        print(
            f"\nNo samples found for {title}"
        )

        return None


    print(
        f"\nCreating grid: {title}"
    )

    print(
        f"Samples: {rows}"
    )


    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        rows,
        3,
        figsize=(
            12,
            max(
                3.2 * rows,
                4
            )
        )
    )


    # Handle one-row case
    if rows == 1:

        axes = np.expand_dims(
            axes,
            axis=0
        )


    # --------------------------------------------------------
    # Process samples
    # --------------------------------------------------------

    for row, (_, record) in enumerate(
        df.iterrows()
    ):

        dataset_index = int(
            record["dataset_index"]
        )

        true_label = int(
            record["true_label"]
        )

        pred224 = int(
            record["pred_224"]
        )

        pred448 = int(
            record["pred_448"]
        )


        # ----------------------------------------------------
        # Get transformed images
        # ----------------------------------------------------

        image_224, label_224 = (
            val_dataset[
                dataset_index
            ]
        )

        image_448, label_448 = (
            val_dataset_448[
                dataset_index
            ]
        )


        # ----------------------------------------------------
        # Verify labels
        # ----------------------------------------------------

        assert (
            label_224
            == true_label
        )

        assert (
            label_448
            == true_label
        )


        # ----------------------------------------------------
        # Prepare inputs
        # ----------------------------------------------------

        input_224 = (
            image_224
            .unsqueeze(0)
            .to(device)
        )

        input_448 = (
            image_448
            .unsqueeze(0)
            .to(device)
        )


        # ----------------------------------------------------
        # Generate Grad-CAM
        # ----------------------------------------------------

        cam224_map = cam_224.generate(
            input_224,
            pred224
        )

        cam448_map = cam_448.generate(
            input_448,
            pred448
        )


        # ----------------------------------------------------
        # Original image
        # ImageFolder stores:
        # val_dataset.samples[index][0]
        # ----------------------------------------------------

        raw_path = (
            val_dataset
            .samples[
                dataset_index
            ][0]
        )

        raw_image = (
            Image.open(
                raw_path
            )
            .convert("L")
        )

        raw_image = np.array(
            raw_image
        )


        # ----------------------------------------------------
        # Column 1 — ORIGINAL
        # ----------------------------------------------------

        axes[row, 0].imshow(
            raw_image,
            cmap="gray"
        )

        axes[row, 0].set_title(
            f"Original | True KL1\n"
            f"Index {dataset_index}",
            fontsize=9
        )

        axes[row, 0].axis("off")


        # ----------------------------------------------------
        # Column 2 — 224 Grad-CAM
        # ----------------------------------------------------

        plot_cam(
            axes[row, 1],
            image_224,
            cam224_map,
            f"224×224 → KL{pred224}"
        )


        # ----------------------------------------------------
        # Column 3 — 448 Grad-CAM
        # ----------------------------------------------------

        plot_cam(
            axes[row, 2],
            image_448,
            cam448_map,
            f"448×448 → KL{pred448}"
        )


    # ========================================================
    # Save figure
    # ========================================================

    fig.suptitle(
        title,
        fontsize=14,
        y=0.995
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.99
        ]
    )

    save_path = (
        "/content/"
        + filename
    )

    plt.savefig(
        save_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)

    print(
        "Saved:",
        save_path
    )

    return save_path


# ============================================================
# 14. CREATE IMPROVED GRID
# ============================================================

improved_path = create_transition_grid(
    improved_df,
    "KL1_224_wrong_448_correct.png",
    "KL1: 224 Wrong → 448 Correct"
)


# ============================================================
# 15. CREATE REGRESSION GRID
# ============================================================

regressed_path = create_transition_grid(
    regressed_df,
    "KL1_224_correct_448_wrong.png",
    "KL1: 224 Correct → 448 Wrong"
)


# ============================================================
# 16. CLEANUP
# ============================================================

cam_224.close()
cam_448.close()


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("VISUAL COMPARISON COMPLETE")
print("=" * 70)

print(
    "\nComparison CSV:",
    comparison_path
)

print(
    "Improved grid:",
    improved_path
)

print(
    "Regression grid:",
    regressed_path
)

print("\nKL1 transition counts:")
print(
    comparison_df[
        comparison_df["true_label"] == 1
    ]["transition"]
    .value_counts()
    .to_string()
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 31 — CORN MODEL SETUP | 448x448 | SEED=2026
# ============================================================

import torch
import torch.nn as nn

from torchvision import models
from coral_pytorch.losses import corn_loss


# ============================================================
# 1. Re-apply seed
# ============================================================

SEED_CORN = 2026

torch.manual_seed(SEED_CORN)
torch.cuda.manual_seed_all(SEED_CORN)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# 2. Create pretrained ResNet18
# ============================================================

corn_model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)


# ============================================================
# 3. Replace final layer
# CORN uses num_classes - 1 outputs
# ============================================================

num_features_corn = (
    corn_model.fc.in_features
)

corn_model.fc = nn.Linear(
    num_features_corn,
    5 - 1
)


# ============================================================
# 4. Move to GPU
# ============================================================

corn_model = corn_model.to(device)


# ============================================================
# 5. Optimizer
# ============================================================

corn_optimizer = torch.optim.Adam(
    corn_model.parameters(),
    lr=1e-4,
    weight_decay=0
)


# ============================================================
# 6. Test forward pass
# ============================================================

dummy_input_corn = torch.randn(
    2,
    3,
    448,
    448,
    device=device
)

with torch.no_grad():

    dummy_logits_corn = (
        corn_model(
            dummy_input_corn
        )
    )


print("=" * 70)
print("CORN MODEL CHECK")
print("=" * 70)

print(
    "Seed:",
    SEED_CORN
)

print(
    "Input shape:",
    dummy_input_corn.shape
)

print(
    "Output shape:",
    dummy_logits_corn.shape
)

print(
    "Expected output:",
    "(2, 4)"
)

print(
    "Loss: corn_loss"
)

print(
    "Optimizer: Adam"
)

print(
    "Learning rate: 1e-4"
)

print(
    "Weight decay: 0"
)

print(
    "Device:",
    device
)


# ============================================================
# 7. Test CORN loss
# ============================================================

dummy_labels_corn = torch.tensor(
    [0, 3],
    dtype=torch.long,
    device=device
)

dummy_loss_corn = corn_loss(
    dummy_logits_corn,
    dummy_labels_corn,
    num_classes=5
)

print(
    "\nTest CORN loss:",
    dummy_loss_corn.item()
)


# ============================================================
# 8. Assertions
# ============================================================

assert dummy_logits_corn.shape == (
    2,
    4
)

assert torch.isfinite(
    dummy_loss_corn
)

print(
    "\n" + "=" * 70
)

print(
    "CORN MODEL CHECK PASSED"
)

print(
    "=" * 70
)

In [ ]:
# ============================================================
# CELL 32 — TRAIN CORN | 448x448 | SEED=2026
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    cohen_kappa_score
)

from coral_pytorch.losses import corn_loss


# ============================================================
# 1. Configuration
# ============================================================

NUM_EPOCHS_CORN = 10
NUM_CLASSES_CORN = 5

best_macro_f1_corn = -1.0
best_epoch_corn = -1
best_state_dict_corn = None

history_corn = []


# ============================================================
# 2. CORN prediction function
#
# For K classes, CORN produces K-1 logits:
# logit_0 -> P(y > 0)
# logit_1 -> P(y > 1)
# logit_2 -> P(y > 2)
# logit_3 -> P(y > 3)
#
# We decode by counting thresholds that are passed.
# ============================================================

def corn_predict(
    logits
):

    probabilities = torch.sigmoid(
        logits
    )

    predictions = (
        probabilities > 0.5
    ).sum(
        dim=1
    )

    return predictions.long()


# ============================================================
# 3. Validation function
# ============================================================

def evaluate_corn(
    model,
    loader
):

    model.eval()

    all_true = []
    all_pred = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = model(
                images
            )

            loss = corn_loss(
                logits,
                labels,
                num_classes=NUM_CLASSES_CORN
            )

            batch_size = (
                images.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )

            total_samples += (
                batch_size
            )

            predictions = corn_predict(
                logits
            )

            all_true.extend(
                labels.cpu().numpy()
            )

            all_pred.extend(
                predictions.cpu().numpy()
            )


    avg_loss = (
        total_loss
        / total_samples
    )

    all_true = np.asarray(
        all_true
    )

    all_pred = np.asarray(
        all_pred
    )


    # ========================================================
    # Metrics
    # ========================================================

    accuracy = accuracy_score(
        all_true,
        all_pred
    )

    balanced_acc = (
        balanced_accuracy_score(
            all_true,
            all_pred
        )
    )

    macro_f1 = f1_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0
    )

    mae = mean_absolute_error(
        all_true,
        all_pred
    )

    qwk = cohen_kappa_score(
        all_true,
        all_pred,
        weights="quadratic"
    )

    adjacent_acc = np.mean(
        np.abs(
            all_true
            -
            all_pred
        ) <= 1
    )

    per_class_f1 = f1_score(
        all_true,
        all_pred,
        labels=[
            0,
            1,
            2,
            3,
            4
        ],
        average=None,
        zero_division=0
    )

    per_class_recall = []

    for class_id in range(
        NUM_CLASSES_CORN
    ):

        mask = (
            all_true == class_id
        )

        if mask.sum() == 0:

            recall = 0.0

        else:

            recall = np.mean(
                all_pred[mask]
                == class_id
            )

        per_class_recall.append(
            recall
        )

    cm = confusion_matrix(
        all_true,
        all_pred,
        labels=[
            0,
            1,
            2,
            3,
            4
        ]
    )

    return {

        "loss": avg_loss,

        "accuracy": accuracy,

        "balanced_acc":
            balanced_acc,

        "macro_f1":
            macro_f1,

        "mae": mae,

        "qwk": qwk,

        "adjacent_acc":
            adjacent_acc,

        "per_class_f1":
            np.asarray(
                per_class_f1
            ),

        "per_class_recall":
            np.asarray(
                per_class_recall
            ),

        "confusion_matrix":
            cm,

        "true":
            all_true,

        "pred":
            all_pred
    }


# ============================================================
# 4. Training
# ============================================================

print("=" * 70)
print("STARTING CORN TRAINING | 448x448 | SEED=2026")
print("=" * 70)

for epoch in range(
    1,
    NUM_EPOCHS_CORN + 1
):

    corn_model.train()

    running_loss = 0.0
    sample_count = 0


    # --------------------------------------------------------
    # Training batches
    # --------------------------------------------------------

    for images, labels in (
        train_loader_448
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        corn_optimizer.zero_grad(
            set_to_none=True
        )

        logits = corn_model(
            images
        )

        loss = corn_loss(
            logits,
            labels,
            num_classes=NUM_CLASSES_CORN
        )

        loss.backward()

        corn_optimizer.step()

        batch_size = (
            images.size(0)
        )

        running_loss += (
            loss.item()
            * batch_size
        )

        sample_count += (
            batch_size
        )


    train_loss = (
        running_loss
        / sample_count
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_results_corn = (
        evaluate_corn(
            corn_model,
            val_loader_448
        )
    )


    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------

    history_corn.append({

        "epoch": epoch,

        "train_loss":
            train_loss,

        "val_loss":
            val_results_corn[
                "loss"
            ],

        "val_accuracy":
            val_results_corn[
                "accuracy"
            ],

        "val_balanced_acc":
            val_results_corn[
                "balanced_acc"
            ],

        "val_macro_f1":
            val_results_corn[
                "macro_f1"
            ],

        "val_mae":
            val_results_corn[
                "mae"
            ],

        "val_qwk":
            val_results_corn[
                "qwk"
            ],

        "val_adjacent_acc":
            val_results_corn[
                "adjacent_acc"
            ]
    })


    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS_CORN} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: "
        f"{val_results_corn['loss']:.4f} | "
        f"Acc: "
        f"{val_results_corn['accuracy']:.4f} | "
        f"Bal Acc: "
        f"{val_results_corn['balanced_acc']:.4f} | "
        f"Macro F1: "
        f"{val_results_corn['macro_f1']:.4f} | "
        f"MAE: "
        f"{val_results_corn['mae']:.4f} | "
        f"QWK: "
        f"{val_results_corn['qwk']:.4f} | "
        f"Adjacent: "
        f"{val_results_corn['adjacent_acc']:.4f}"
    )


    # --------------------------------------------------------
    # Best model by Macro F1
    # --------------------------------------------------------

    if (
        val_results_corn[
            "macro_f1"
        ]
        >
        best_macro_f1_corn
    ):

        best_macro_f1_corn = (
            val_results_corn[
                "macro_f1"
            ]
        )

        best_epoch_corn = epoch

        best_state_dict_corn = {

            key: value.detach()
            .cpu()
            .clone()

            for key, value
            in corn_model
            .state_dict()
            .items()
        }

        print(
            f"  --> New BEST Macro F1: "
            f"{best_macro_f1_corn:.4f}"
        )


# ============================================================
# 5. Restore best model
# ============================================================

corn_model.load_state_dict(
    best_state_dict_corn
)

corn_model = corn_model.to(
    device
)


# ============================================================
# 6. Final validation
# ============================================================

best_results_corn = (
    evaluate_corn(
        corn_model,
        val_loader_448
    )
)


# ============================================================
# 7. Final report
# ============================================================

print("\n" + "=" * 70)
print("BEST CORN VALIDATION RESULT")
print("=" * 70)

print(
    f"Best epoch       : "
    f"{best_epoch_corn}"
)

print(
    f"Accuracy         : "
    f"{best_results_corn['accuracy']:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{best_results_corn['balanced_acc']:.4f}"
)

print(
    f"Macro F1         : "
    f"{best_results_corn['macro_f1']:.4f}"
)

print(
    f"MAE              : "
    f"{best_results_corn['mae']:.4f}"
)

print(
    f"QWK              : "
    f"{best_results_corn['qwk']:.4f}"
)

print(
    f"Adjacent Accuracy: "
    f"{best_results_corn['adjacent_acc']:.4f}"
)


# ============================================================
# 8. Per-class results
# ============================================================

print("\n" + "=" * 70)
print("PER-CLASS RESULTS")
print("=" * 70)

for class_id in range(
    NUM_CLASSES_CORN
):

    print(
        f"KL{class_id}: "
        f"F1="
        f"{best_results_corn['per_class_f1'][class_id]:.4f} | "
        f"Recall="
        f"{best_results_corn['per_class_recall'][class_id]:.4f}"
    )


# ============================================================
# 9. Confusion matrix
# ============================================================

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(
    best_results_corn[
        "confusion_matrix"
    ]
)


# ============================================================
# 10. Save history
# ============================================================

history_corn_df = pd.DataFrame(
    history_corn
)

history_corn_path = (
    "/content/"
    "corn_448_history_seed2026.csv"
)

history_corn_df.to_csv(
    history_corn_path,
    index=False
)


# ============================================================
# 11. Save checkpoint
# ============================================================

corn_checkpoint_path = (
    "/content/"
    "best_resnet18_corn_448_seed2026.pth"
)

torch.save(
    {
        "model_state_dict":
            corn_model.state_dict(),

        "best_epoch":
            best_epoch_corn,

        "macro_f1":
            best_results_corn[
                "macro_f1"
            ],

        "balanced_acc":
            best_results_corn[
                "balanced_acc"
            ],

        "accuracy":
            best_results_corn[
                "accuracy"
            ],

        "mae":
            best_results_corn[
                "mae"
            ],

        "qwk":
            best_results_corn[
                "qwk"
            ],

        "adjacent_acc":
            best_results_corn[
                "adjacent_acc"
            ],

        "seed":
            SEED_CORN,

        "resolution":
            448
    },
    corn_checkpoint_path
)


print("\nSaved:")
print(history_corn_path)
print(corn_checkpoint_path)

print("=" * 70)